In [2]:
import pandas as pd

# headers = requests.utils.default_headers()
# headers.update({
#     'User-Agent': 'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:52.0) Gecko/20100101 Firefox/52.0',
# })

# pmd_24_25_harvest_link = requests.get("https://news.maryland.gov/dnr/2025/02/14/maryland-hunters-harvest-84201-deer-for-2024-2025-season/", headers=headers)
# pmd_24_25_harvest_content = BeautifulSoup(pmd_24_25_harvest_link.text, 'html.parser')

#https://stackoverflow.com/questions/43440397/requests-using-beautiful-soup-gets-blocked 

In [3]:
#2025-2026 Processed Harvest Data pt. 1
df_pmd_25_26 = pd.read_html("https://news.maryland.gov/dnr/2026/02/12/maryland-hunters-harvest-71649-deer-for-2025-2026-season/", storage_options={"User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:52.0) Gecko/20100101 Firefox/52.0"})
df_pmd_25_26 = df_pmd_25_26[0]
#type(df_pmd_25_26_one)
df_pmd_25_26.to_csv("df_pmd_25_26.csv")

In [4]:
#2025-2026 Processed Harvest Data pt. 2 - COMBINING HEADERS

header_row_two_2526 = df_pmd_25_26.iloc[1]
header_row_three_2526 = df_pmd_25_26.iloc[2]

#first row iloc[0] is title of the table 
#second row is antlered, anterless, and total
#third row is previous hunting season, the hunting season that just concluded, percentage change in harvest
#this third row is grouped under antlered, anterless, and total 

#because I want to combine second and third rows, start by create variables that extract values in second and third rows

#this for loop creates a list of strings, which are the combined header names, which will then be applied to the dataframe as names for the new column
new_columns = []
#need to create an empty list in order to store the combined column names (combining current names of rows 2 and 3)

#this for loop loops through both header rows at the same time, since the change needs to apply to both 
#zip() pairs values from the two rows column-by-column 
for header_two, header_three in zip(header_row_two_2526, header_row_three_2526):
    if pd.isna(header_two):
    #https://pandas.pydata.org/docs/reference/api/pandas.isna.html 
        #using isna because there's a cell above county (appears in third row) that's empty in the second row. 
        #this is saying if the upper header is missing (header_two) then just use the name in header_three, the lower header 
        combined_header = header_three 
    else:
        combined_header = f"{header_two}_{header_three}"
    new_columns.append(combined_header)

#replaces dataframe's current column names with new combined names stored in new_columns
df_pmd_25_26.columns = new_columns

#removes the header rows from the data 
#colon is python's slicing operator. It's keeping all rows starting at position 3 (which is position of the new column names)
#discarding rows 0-2, which were used for the old header names and the table title, which was repeated across all row cells 
df_pmd_25_26 = df_pmd_25_26.iloc[3:]
#https://www.datacamp.com/tutorial/loc-vs-iloc

df_pmd_25_26 = df_pmd_25_26.reset_index(drop=True)
#reset index after removing old header rows
#drop=True gets rid of old index, preventing it from taking up space as a meaningless column 

#SAVE TO CSV (PROCESSED DATA FOLDER)
df_pmd_25_26.to_csv("df_pmd_25_26.csv")

In [5]:
#---------2025-2026 MORE CLEANING: REPLACING * , DROPPING LAST ROW, CHANGING DATATYPES, DROPPING % COLUMNS---------#

#1. Replace asterisks with NaN
import numpy as np
df_pmd_25_26=df_pmd_25_26.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

#2. Drop last row, which just has "small sample size" text
df_pmd_25_26 = df_pmd_25_26[:-1]

#3. Change all datatypes (except those in county) to floats 
#found this on stack overflow to change datatypes of all but one column 

ignore = ['County']

#NEED TO SAVE THIS TO THE CURRENT DATAFRAME WITH DF=
df_pmd_25_26 = (df_pmd_25_26.set_index(ignore, append=True)
        .astype(float)
        .reset_index(ignore)
       )

#df_pmd_25_26.info()

#https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

#4.Drop percentage columns and then recalculate later 
df_pmd_25_26=df_pmd_25_26.drop(["Antlered_% Change", "Antlerless_% Change", "Total_% Change"], axis=1)

In [6]:
#---------2025-2026 MORE CLEANING: TEST TRIAL FOR COMBINING SUMS---------#
#5. Add whitetail and sika counts for each county manually, and then fill those sums into currently NaN county row 
caroline_whitetail = df_pmd_25_26.iloc[5]
caroline_sika = df_pmd_25_26.iloc[6] 

df_pmd_25_26.iloc[4, 1] = df_pmd_25_26.iloc[[5, 6], 1].sum()
df_pmd_25_26.iloc[4, 2] = df_pmd_25_26.iloc[[5,6], 2].sum()
df_pmd_25_26.iloc[4, 3] = df_pmd_25_26.iloc[[5, 6], 3].sum()
df_pmd_25_26.iloc[4, 4] = df_pmd_25_26.iloc[[5,6], 4].sum()
df_pmd_25_26.iloc[4, 5]= df_pmd_25_26.iloc[[5, 6], 5].sum()
df_pmd_25_26.iloc[4, 6] = df_pmd_25_26.iloc[[5,6], 6].sum()

df_pmd_25_26

,County,Antlered_2024-25,Antlered_2025-26,Antlerless_2024-25,Antlerless_2025-26,Total_2024-25,Total_2025-26
0,Allegany,1868.0,1739.0,1544.0,1185.0,3412.0,2924.0
1,Anne Arundel,898.0,579.0,1371.0,843.0,2269.0,1422.0
2,Baltimore,2109.0,2008.0,3130.0,2771.0,5239.0,4779.0
3,Calvert,658.0,524.0,1062.0,700.0,1720.0,1224.0
4,Caroline,880.0,916.0,2264.0,1624.0,3144.0,2540.0
5,whitetail,878.0,914.0,2262.0,1622.0,3140.0,2536.0
6,sika,2.0,2.0,2.0,2.0,4.0,4.0
7,Carroll,2368.0,2340.0,3620.0,2928.0,5988.0,5268.0
8,Cecil,1325.0,1320.0,2260.0,1839.0,3585.0,3159.0
9,Charles,1356.0,873.0,2012.0,1222.0,3368.0,2095.0


In [7]:
#---------2025-2026 MORE CLEANING: COMBINING ALL SUMS FOR WHITETAIL AND SIKA FOR ALL APPLICABLE COUNTIES---------#

#Dorchester
dorchester_whitetail = df_pmd_25_26.iloc[11]
dorchester_sika = df_pmd_25_26.iloc[12]

df_pmd_25_26.iloc[10, 1] = df_pmd_25_26.iloc[[11, 12], 1].sum()
df_pmd_25_26.iloc[10, 2] = df_pmd_25_26.iloc[[11,12], 2].sum()
df_pmd_25_26.iloc[10, 3] = df_pmd_25_26.iloc[[11, 12], 3].sum()
df_pmd_25_26.iloc[10, 4] = df_pmd_25_26.iloc[[11,12], 4].sum()
df_pmd_25_26.iloc[10, 5] = df_pmd_25_26.iloc[[11, 12], 5].sum()
df_pmd_25_26.iloc[10, 6] = df_pmd_25_26.iloc[[11,12], 6].sum()

#Somerset
somerset_whitetail = df_pmd_25_26.iloc[23]
somerset_sika = df_pmd_25_26.iloc[24]

df_pmd_25_26.iloc[22, 1] = df_pmd_25_26.iloc[[23, 24], 1].sum()
df_pmd_25_26.iloc[22, 2] = df_pmd_25_26.iloc[[23,24], 2].sum()
df_pmd_25_26.iloc[22, 3] = df_pmd_25_26.iloc[[23, 24], 3].sum()
df_pmd_25_26.iloc[22, 4] = df_pmd_25_26.iloc[[23,24], 4].sum()
df_pmd_25_26.iloc[22, 5] = df_pmd_25_26.iloc[[23, 24], 5].sum()
df_pmd_25_26.iloc[22, 6] = df_pmd_25_26.iloc[[23,24], 6].sum()

# #Wicomico
wicomico_whitetail = df_pmd_25_26.iloc[28]
wicomico_sika = df_pmd_25_26.iloc[29]

df_pmd_25_26.iloc[27, 1] = df_pmd_25_26.iloc[[28, 29], 1].sum()
df_pmd_25_26.iloc[27, 2] = df_pmd_25_26.iloc[[28,29], 2].sum()
df_pmd_25_26.iloc[27, 3] = df_pmd_25_26.iloc[[28, 29], 3].sum()
df_pmd_25_26.iloc[27, 4] = df_pmd_25_26.iloc[[28,29], 4].sum()
df_pmd_25_26.iloc[27, 5] = df_pmd_25_26.iloc[[28, 29], 5].sum()
df_pmd_25_26.iloc[27, 6] = df_pmd_25_26.iloc[[28,29], 6].sum()

# #Worcester
worcester_whitetail = df_pmd_25_26.iloc[31]
worcester_sika = df_pmd_25_26.iloc[32]

df_pmd_25_26.iloc[30, 1] = df_pmd_25_26.iloc[[31, 32], 1].sum()
df_pmd_25_26.iloc[30, 2] = df_pmd_25_26.iloc[[31,32], 2].sum()
df_pmd_25_26.iloc[30, 3] = df_pmd_25_26.iloc[[31, 32], 3].sum()
df_pmd_25_26.iloc[30, 4] = df_pmd_25_26.iloc[[31,32], 4].sum()
df_pmd_25_26.iloc[30, 5] = df_pmd_25_26.iloc[[31, 32], 5].sum()
df_pmd_25_26.iloc[30, 6] = df_pmd_25_26.iloc[[31,32], 6].sum()

df_pmd_25_26

,County,Antlered_2024-25,Antlered_2025-26,Antlerless_2024-25,Antlerless_2025-26,Total_2024-25,Total_2025-26
0,Allegany,1868.0,1739.0,1544.0,1185.0,3412.0,2924.0
1,Anne Arundel,898.0,579.0,1371.0,843.0,2269.0,1422.0
2,Baltimore,2109.0,2008.0,3130.0,2771.0,5239.0,4779.0
3,Calvert,658.0,524.0,1062.0,700.0,1720.0,1224.0
4,Caroline,880.0,916.0,2264.0,1624.0,3144.0,2540.0
5,whitetail,878.0,914.0,2262.0,1622.0,3140.0,2536.0
6,sika,2.0,2.0,2.0,2.0,4.0,4.0
7,Carroll,2368.0,2340.0,3620.0,2928.0,5988.0,5268.0
8,Cecil,1325.0,1320.0,2260.0,1839.0,3585.0,3159.0
9,Charles,1356.0,873.0,2012.0,1222.0,3368.0,2095.0


In [8]:
#---------2025-2026 FINAL CLEANING: DROPPING ROWS---------#
#Filter county values for "whitetail" and "sika" and drop those rows. Also drop "Total"
df_pmd_25_26=df_pmd_25_26[ df_pmd_25_26[ "County" ].str.contains( "whitetail|sika" )== False]
df_pmd_25_26 = df_pmd_25_26.reset_index(drop=True)
df_pmd_25_26.to_csv("df_pmd_25_26.csv")

In [9]:
#2024-2025 Processed Harvest Data pt. 1 

#https://pandas.pydata.org/docs/reference/api/pandas.read_html.html
#https://stackoverflow.com/questions/43590153/http-error-403-forbidden-when-reading-html .

df_pmd_24_25 = pd.read_html("https://news.maryland.gov/dnr/2025/02/14/maryland-hunters-harvest-84201-deer-for-2024-2025-season/", storage_options={"User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:52.0) Gecko/20100101 Firefox/52.0"})
#read_html initially returned list of dataframes, so turn the first item in the list into a df 
    #https://stackoverflow.com/questions/39710903/pd-read-html-imports-a-list-rather-than-a-dataframe
df_pmd_24_25 = df_pmd_24_25[0]
df_pmd_24_25.to_csv("df_pmd_24_25.csv")

In [10]:
#2024-2025 Processed Harvest Data pt. 2 = COMBINING HEADERS
header_row_one_2425 = df_pmd_24_25.iloc[1]
header_row_two_2425 = df_pmd_24_25.iloc[2]

new_columns = []

for header_one, header_two in zip(header_row_one_2425, header_row_two_2425):
    if pd.isna(header_one):
    #https://pandas.pydata.org/docs/reference/api/pandas.isna.html 
        combined_header = header_two 
    else:
        combined_header = f"{header_one}_{header_two}"
    new_columns.append(combined_header)
    
df_pmd_24_25.columns = new_columns 

df_pmd_24_25 = df_pmd_24_25.iloc[3:]
#https://www.datacamp.com/tutorial/loc-vs-iloc

df_pmd_24_25 = df_pmd_24_25.reset_index(drop=True)

df_pmd_24_25

,County,Antlered_2023-24,Antlered_2024-25,Antlered_% Change,Antlerless_2023-24,Antlerless_2024-25,Antlerless_% Change,Total_2023-24,Total_2024-25,Total_% Change
0,Allegany,1824,1868,2.4,1128,1544,36.9,2952,3412,15.6
1,Anne Arundel,750,898,19.7,974,1371,40.8,1724,2269,31.6
2,Baltimore,1899,2109,11.1,2651,3130,18.1,4550,5239,15.1
3,Calvert,584,658,12.7,817,1062,30.0,1401,1720,22.8
4,Caroline,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,whitetail,910,878,-3.5,1666,2262,35.8,2576,3140,21.9
6,sika,0,2,*,1,2,*,1,4,*
7,Carroll,2470,2368,-4.1,3259,3620,11.1,5729,5988,4.5
8,Cecil,1177,1325,12.6,1844,2260,22.6,3021,3585,18.7
9,Charles,1050,1356,29.1,1383,2012,45.5,2433,3368,38.4


In [11]:
#---------2024-2025 MORE CLEANING: REPLACING * , DROPPING LAST ROW, CHANGING DATATYPES, DROPPING % COLUMNS---------#

#1. Replace asterisks with NaN
import numpy as np
df_pmd_24_25=df_pmd_24_25.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

#2. Drop last row, which just has "small sample size" text
df_pmd_24_25 = df_pmd_24_25.iloc[:-1]

#3. Change all datatypes (except those in county) to floats 
#found this on stack overflow to change datatypes of all but one column 

ignore = ['County']

#NEED TO SAVE THIS TO THE CURRENT DATAFRAME WITH DF=
df_pmd_24_25 = (df_pmd_24_25.set_index(ignore, append=True)
        .astype(float)
        .reset_index(ignore)
       )

# df_pmd_24_25.info()

#https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

#4.Drop percentage columns and then recalculate later 
df_pmd_24_25=df_pmd_24_25.drop(["Antlered_% Change", "Antlerless_% Change", "Total_% Change"], axis=1)
df_pmd_24_25

,County,Antlered_2023-24,Antlered_2024-25,Antlerless_2023-24,Antlerless_2024-25,Total_2023-24,Total_2024-25
0,Allegany,1824.0,1868.0,1128.0,1544.0,2952.0,3412.0
1,Anne Arundel,750.0,898.0,974.0,1371.0,1724.0,2269.0
2,Baltimore,1899.0,2109.0,2651.0,3130.0,4550.0,5239.0
3,Calvert,584.0,658.0,817.0,1062.0,1401.0,1720.0
4,Caroline,NaN,NaN,NaN,NaN,NaN,NaN
5,whitetail,910.0,878.0,1666.0,2262.0,2576.0,3140.0
6,sika,0.0,2.0,1.0,2.0,1.0,4.0
7,Carroll,2470.0,2368.0,3259.0,3620.0,5729.0,5988.0
8,Cecil,1177.0,1325.0,1844.0,2260.0,3021.0,3585.0
9,Charles,1050.0,1356.0,1383.0,2012.0,2433.0,3368.0


In [12]:
#---------2024-2025 MORE CLEANING: TEST TRIAL FOR COMBINING SUMS---------#
#5. Add whitetail and sika counts for each county manually, and then fill those sums into currently NaN county row 

#CAROLINE COUNTY
df_pmd_24_25.iloc[4, 1] = df_pmd_24_25.iloc[[5, 6], 1].sum()
df_pmd_24_25.iloc[4, 2] = df_pmd_24_25.iloc[[5,6], 2].sum()
df_pmd_24_25.iloc[4, 3] = df_pmd_24_25.iloc[[5, 6], 3].sum()
df_pmd_24_25.iloc[4, 4] = df_pmd_24_25.iloc[[5,6], 4].sum()
df_pmd_24_25.iloc[4, 5]= df_pmd_24_25.iloc[[5, 6], 5].sum()
df_pmd_24_25.iloc[4, 6] = df_pmd_24_25.iloc[[5,6], 6].sum()

df_pmd_24_25

,County,Antlered_2023-24,Antlered_2024-25,Antlerless_2023-24,Antlerless_2024-25,Total_2023-24,Total_2024-25
0,Allegany,1824.0,1868.0,1128.0,1544.0,2952.0,3412.0
1,Anne Arundel,750.0,898.0,974.0,1371.0,1724.0,2269.0
2,Baltimore,1899.0,2109.0,2651.0,3130.0,4550.0,5239.0
3,Calvert,584.0,658.0,817.0,1062.0,1401.0,1720.0
4,Caroline,910.0,880.0,1667.0,2264.0,2577.0,3144.0
5,whitetail,910.0,878.0,1666.0,2262.0,2576.0,3140.0
6,sika,0.0,2.0,1.0,2.0,1.0,4.0
7,Carroll,2470.0,2368.0,3259.0,3620.0,5729.0,5988.0
8,Cecil,1177.0,1325.0,1844.0,2260.0,3021.0,3585.0
9,Charles,1050.0,1356.0,1383.0,2012.0,2433.0,3368.0


In [13]:
#---------2024-2025 MORE CLEANING: COMBINING ALL SUMS FOR WHITETAIL AND SIKA FOR ALL APPLICABLE COUNTIES---------#

# #Dorchester
df_pmd_24_25.iloc[10, 1] = df_pmd_24_25.iloc[[11, 12], 1].sum()
df_pmd_24_25.iloc[10, 2] = df_pmd_24_25.iloc[[11,12], 2].sum()
df_pmd_24_25.iloc[10, 3] = df_pmd_24_25.iloc[[11, 12], 3].sum()
df_pmd_24_25.iloc[10, 4] = df_pmd_24_25.iloc[[11,12], 4].sum()
df_pmd_24_25.iloc[10, 5] = df_pmd_24_25.iloc[[11, 12], 5].sum()
df_pmd_24_25.iloc[10, 6] = df_pmd_24_25.iloc[[11,12], 6].sum()

# #Somerset
df_pmd_24_25.iloc[22, 1] = df_pmd_24_25.iloc[[23, 24], 1].sum()
df_pmd_24_25.iloc[22, 2] = df_pmd_24_25.iloc[[23,24], 2].sum()
df_pmd_24_25.iloc[22, 3] = df_pmd_24_25.iloc[[23, 24], 3].sum()
df_pmd_24_25.iloc[22, 4] = df_pmd_24_25.iloc[[23,24], 4].sum()
df_pmd_24_25.iloc[22, 5] = df_pmd_24_25.iloc[[23, 24], 5].sum()
df_pmd_24_25.iloc[22, 6] = df_pmd_24_25.iloc[[23,24], 6].sum()

# # #Wicomico
df_pmd_24_25.iloc[27, 1] = df_pmd_24_25.iloc[[28, 29], 1].sum()
df_pmd_24_25.iloc[27, 2] = df_pmd_24_25.iloc[[28,29], 2].sum()
df_pmd_24_25.iloc[27, 3] = df_pmd_24_25.iloc[[28, 29], 3].sum()
df_pmd_24_25.iloc[27, 4] = df_pmd_24_25.iloc[[28,29], 4].sum()
df_pmd_24_25.iloc[27, 5] = df_pmd_24_25.iloc[[28, 29], 5].sum()
df_pmd_24_25.iloc[27, 6] = df_pmd_24_25.iloc[[28,29], 6].sum()

# # #Worcester
df_pmd_24_25.iloc[30, 1] = df_pmd_24_25.iloc[[31, 32], 1].sum()
df_pmd_24_25.iloc[30, 2] = df_pmd_24_25.iloc[[31,32], 2].sum()
df_pmd_24_25.iloc[30, 3] = df_pmd_24_25.iloc[[31, 32], 3].sum()
df_pmd_24_25.iloc[30, 4] = df_pmd_24_25.iloc[[31,32], 4].sum()
df_pmd_24_25.iloc[30, 5] = df_pmd_24_25.iloc[[31, 32], 5].sum()
df_pmd_24_25.iloc[30, 6] = df_pmd_24_25.iloc[[31,32], 6].sum()

df_pmd_24_25

,County,Antlered_2023-24,Antlered_2024-25,Antlerless_2023-24,Antlerless_2024-25,Total_2023-24,Total_2024-25
0,Allegany,1824.0,1868.0,1128.0,1544.0,2952.0,3412.0
1,Anne Arundel,750.0,898.0,974.0,1371.0,1724.0,2269.0
2,Baltimore,1899.0,2109.0,2651.0,3130.0,4550.0,5239.0
3,Calvert,584.0,658.0,817.0,1062.0,1401.0,1720.0
4,Caroline,910.0,880.0,1667.0,2264.0,2577.0,3144.0
5,whitetail,910.0,878.0,1666.0,2262.0,2576.0,3140.0
6,sika,0.0,2.0,1.0,2.0,1.0,4.0
7,Carroll,2470.0,2368.0,3259.0,3620.0,5729.0,5988.0
8,Cecil,1177.0,1325.0,1844.0,2260.0,3021.0,3585.0
9,Charles,1050.0,1356.0,1383.0,2012.0,2433.0,3368.0


In [14]:
#---------2024-2025 FINAL CLEANING: DROPPING ROWS---------#
#Filter county values for "whitetail" and "sika" and drop those rows
df_pmd_24_25=df_pmd_24_25[ df_pmd_24_25[ "County" ].str.contains( "whitetail|sika" )== False]
df_pmd_24_25 = df_pmd_24_25.reset_index(drop=True)
df_pmd_24_25.to_csv("df_pmd_24_25.csv")

In [15]:
#2023-2024 Processed Harvest Data Pt. 1

#PANDAS
df_pmd_23_24 = pd.read_html("https://news.maryland.gov/dnr/2024/02/13/maryland-hunters-harvest-72642-deer-for-2023-2024-season/", storage_options={"User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:52.0) Gecko/20100101 Firefox/52.0"})
df_pmd_23_24 = df_pmd_23_24[0]
#type(df_pmd_23_24)
df_pmd_23_24.to_csv("df_pmd_23_24.csv")

In [16]:
#2023-2024 Processed Harvest Data pt. 2 = COMBINING HEADERS

header_row_one_2324 = df_pmd_23_24.iloc[1]
header_row_two_2324 = df_pmd_23_24.iloc[2]

new_columns = []

for header_one, header_two in zip(header_row_one_2324, header_row_two_2324):
    if pd.isna(header_one):
    #https://pandas.pydata.org/docs/reference/api/pandas.isna.html 
        combined_header = header_two 
    else:
        combined_header = f"{header_one}_{header_two}"
    new_columns.append(combined_header)

#STORE NEW COMBINED HEADERS
df_pmd_23_24.columns = new_columns

#REPLACE OLD HEADERS WITH COMBINED
df_pmd_23_24 = df_pmd_23_24.iloc[3:]
df_pmd_23_24 = df_pmd_23_24.reset_index(drop=True)

#df_pmd_23_24

#SAVE TO CSV (PROCESSED DATA FOLDER)
df_pmd_23_24.to_csv("df_pmd_23_24.csv")
df_pmd_23_24

,County,Antlered_2023-24,Antlered_2022-23,Antlered_% Change,Antlerless_2023-24,Antlerless_2022-23,Antlerless_% Change,Total_2023-24,Total_2022-23,Total_% Change
0,Allegany,1824,1925,-5.2,1128,1474,-23.5,2952,3399,-13.2
1,Anne Arundel,750,800,-6.3,974,1083,-10.1,1724,1883,-8.4
2,Baltimore,1899,1856,2.3,2651,2990,-11.3,4550,4846,-6.1
3,Calvert,584,599,-2.5,817,879,-7.1,1401,1478,-5.2
4,Caroline,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,whitetail,910,933,-2.5,1666,1967,-15.3,2576,2900,-11.2
6,sika,0,0,*,1,1,*,1,1,*
7,Carroll,2470,2351,5.1,3259,3434,-5.1,5729,5785,-1
8,Cecil,1177,1230,-4.3,1844,2122,-13.1,3021,3352,-9.9
9,Charles,1050,1136,-7.6,1383,1439,-3.9,2433,2575,-5.5


In [17]:
#---------2023-2024 MORE CLEANING: REPLACING * , DROPPING LAST ROW, CHANGING DATATYPES, DROPPING % COLUMNS---------#

#1. Replace asterisks with NaN
import numpy as np
df_pmd_23_24=df_pmd_23_24.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

#2. Drop last row, which just has "small sample size" text
df_pmd_23_24 = df_pmd_23_24.iloc[:-1]

#3. Change all datatypes (except those in county) to floats 
#found this on stack overflow to change datatypes of all but one column 

ignore = ['County']

#NEED TO SAVE THIS TO THE CURRENT DATAFRAME WITH DF=
df_pmd_23_24 = (df_pmd_23_24.set_index(ignore, append=True)
        .astype(float)
        .reset_index(ignore)
       )

# df_pmd_23_24.info()

#https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

#4.Drop percentage columns and then recalculate later 
df_pmd_23_24=df_pmd_23_24.drop(["Antlered_% Change", "Antlerless_% Change", "Total_% Change"], axis=1)
df_pmd_23_24

,County,Antlered_2023-24,Antlered_2022-23,Antlerless_2023-24,Antlerless_2022-23,Total_2023-24,Total_2022-23
0,Allegany,1824.0,1925.0,1128.0,1474.0,2952.0,3399.0
1,Anne Arundel,750.0,800.0,974.0,1083.0,1724.0,1883.0
2,Baltimore,1899.0,1856.0,2651.0,2990.0,4550.0,4846.0
3,Calvert,584.0,599.0,817.0,879.0,1401.0,1478.0
4,Caroline,NaN,NaN,NaN,NaN,NaN,NaN
5,whitetail,910.0,933.0,1666.0,1967.0,2576.0,2900.0
6,sika,0.0,0.0,1.0,1.0,1.0,1.0
7,Carroll,2470.0,2351.0,3259.0,3434.0,5729.0,5785.0
8,Cecil,1177.0,1230.0,1844.0,2122.0,3021.0,3352.0
9,Charles,1050.0,1136.0,1383.0,1439.0,2433.0,2575.0


In [18]:
#---------2023-2024 MORE CLEANING: TEST TRIAL FOR COMBINING SUMS---------#
#5. Add whitetail and sika counts for each county manually, and then fill those sums into currently NaN county row 

#CAROLINE COUNTY
df_pmd_23_24.iloc[4, 1] = df_pmd_23_24.iloc[[5, 6], 1].sum()
df_pmd_23_24.iloc[4, 2] = df_pmd_23_24.iloc[[5,6], 2].sum()
df_pmd_23_24.iloc[4, 3] = df_pmd_23_24.iloc[[5, 6], 3].sum()
df_pmd_23_24.iloc[4, 4] = df_pmd_23_24.iloc[[5,6], 4].sum()
df_pmd_23_24.iloc[4, 5] = df_pmd_23_24.iloc[[5, 6], 5].sum()
df_pmd_23_24.iloc[4, 6] = df_pmd_23_24.iloc[[5,6], 6].sum()

df_pmd_23_24

,County,Antlered_2023-24,Antlered_2022-23,Antlerless_2023-24,Antlerless_2022-23,Total_2023-24,Total_2022-23
0,Allegany,1824.0,1925.0,1128.0,1474.0,2952.0,3399.0
1,Anne Arundel,750.0,800.0,974.0,1083.0,1724.0,1883.0
2,Baltimore,1899.0,1856.0,2651.0,2990.0,4550.0,4846.0
3,Calvert,584.0,599.0,817.0,879.0,1401.0,1478.0
4,Caroline,910.0,933.0,1667.0,1968.0,2577.0,2901.0
5,whitetail,910.0,933.0,1666.0,1967.0,2576.0,2900.0
6,sika,0.0,0.0,1.0,1.0,1.0,1.0
7,Carroll,2470.0,2351.0,3259.0,3434.0,5729.0,5785.0
8,Cecil,1177.0,1230.0,1844.0,2122.0,3021.0,3352.0
9,Charles,1050.0,1136.0,1383.0,1439.0,2433.0,2575.0


In [19]:
#---------2023-2024 MORE CLEANING: COMBINING ALL SUMS FOR WHITETAIL AND SIKA FOR ALL APPLICABLE COUNTIES---------#

# #Dorchester
df_pmd_23_24.iloc[10, 1] = df_pmd_23_24.iloc[[11, 12], 1].sum()
df_pmd_23_24.iloc[10, 2] = df_pmd_23_24.iloc[[11,12], 2].sum()
df_pmd_23_24.iloc[10, 3] = df_pmd_23_24.iloc[[11, 12], 3].sum()
df_pmd_23_24.iloc[10, 4] = df_pmd_23_24.iloc[[11,12], 4].sum()
df_pmd_23_24.iloc[10, 5] = df_pmd_23_24.iloc[[11, 12], 5].sum()
df_pmd_23_24.iloc[10, 6] = df_pmd_23_24.iloc[[11,12], 6].sum()

# #Somerset
df_pmd_23_24.iloc[22, 1] = df_pmd_23_24.iloc[[23, 24], 1].sum()
df_pmd_23_24.iloc[22, 2] = df_pmd_23_24.iloc[[23,24], 2].sum()
df_pmd_23_24.iloc[22, 3] = df_pmd_23_24.iloc[[23, 24], 3].sum()
df_pmd_23_24.iloc[22, 4] = df_pmd_23_24.iloc[[23,24], 4].sum()
df_pmd_23_24.iloc[22, 5] = df_pmd_23_24.iloc[[23, 24], 5].sum()
df_pmd_23_24.iloc[22, 6] = df_pmd_23_24.iloc[[23,24], 6].sum()

# # #Wicomico
df_pmd_23_24.iloc[27, 1] = df_pmd_23_24.iloc[[28, 29], 1].sum()
df_pmd_23_24.iloc[27, 2] = df_pmd_23_24.iloc[[28,29], 2].sum()
df_pmd_23_24.iloc[27, 3] = df_pmd_23_24.iloc[[28, 29], 3].sum()
df_pmd_23_24.iloc[27, 4] = df_pmd_23_24.iloc[[28,29], 4].sum()
df_pmd_23_24.iloc[27, 5] = df_pmd_23_24.iloc[[28, 29], 5].sum()
df_pmd_23_24.iloc[27, 6] = df_pmd_23_24.iloc[[28,29], 6].sum()

# # #Worcester
df_pmd_23_24.iloc[30, 1] = df_pmd_23_24.iloc[[31, 32], 1].sum()
df_pmd_23_24.iloc[30, 2] = df_pmd_23_24.iloc[[31,32], 2].sum()
df_pmd_23_24.iloc[30, 3] = df_pmd_23_24.iloc[[31, 32], 3].sum()
df_pmd_23_24.iloc[30, 4] = df_pmd_23_24.iloc[[31,32], 4].sum()
df_pmd_23_24.iloc[30, 5] = df_pmd_23_24.iloc[[31, 32], 5].sum()
df_pmd_23_24.iloc[30, 6] = df_pmd_23_24.iloc[[31,32], 6].sum()

df_pmd_23_24

,County,Antlered_2023-24,Antlered_2022-23,Antlerless_2023-24,Antlerless_2022-23,Total_2023-24,Total_2022-23
0,Allegany,1824.0,1925.0,1128.0,1474.0,2952.0,3399.0
1,Anne Arundel,750.0,800.0,974.0,1083.0,1724.0,1883.0
2,Baltimore,1899.0,1856.0,2651.0,2990.0,4550.0,4846.0
3,Calvert,584.0,599.0,817.0,879.0,1401.0,1478.0
4,Caroline,910.0,933.0,1667.0,1968.0,2577.0,2901.0
5,whitetail,910.0,933.0,1666.0,1967.0,2576.0,2900.0
6,sika,0.0,0.0,1.0,1.0,1.0,1.0
7,Carroll,2470.0,2351.0,3259.0,3434.0,5729.0,5785.0
8,Cecil,1177.0,1230.0,1844.0,2122.0,3021.0,3352.0
9,Charles,1050.0,1136.0,1383.0,1439.0,2433.0,2575.0


In [20]:
#---------2023-2024 FINAL CLEANING: DROPPING ROWS---------#
#Filter county values for "whitetail" and "sika" and drop those rows
df_pmd_23_24=df_pmd_23_24[ df_pmd_23_24[ "County" ].str.contains( "whitetail|sika" )== False]
df_pmd_23_24 = df_pmd_23_24.reset_index(drop=True)
df_pmd_23_24.to_csv("df_pmd_23_24.csv")

In [21]:
#2022-2023 Processed Harvest Data Pt. 1

#PANDAS
df_pmd_22_23 = pd.read_html("https://news.maryland.gov/dnr/2023/02/16/maryland-hunters-harvest-76687-deer-for-2022-2023-season/", storage_options={"User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:52.0) Gecko/20100101 Firefox/52.0"})
df_pmd_22_23 = df_pmd_22_23[0]
#type(df_pmd_22_23)
df_pmd_22_23.to_csv("df_pmd_22_23_one.csv")



In [22]:
#2022-2023 Processed Harvest Data pt. 2 = COMBINING HEADERS

header_row_one_2223 = df_pmd_22_23.iloc[1]
header_row_two_2223 = df_pmd_22_23.iloc[2]

new_columns = []

for header_one, header_two in zip(header_row_one_2223, header_row_two_2223):
    if pd.isna(header_one):
    #https://pandas.pydata.org/docs/reference/api/pandas.isna.html 
        combined_header = header_two 
    else:
        combined_header = f"{header_one}_{header_two}"
    new_columns.append(combined_header)

#STORE NEW COMBINED HEADERS
df_pmd_22_23.columns = new_columns

#REPLACE OLD HEADERS WITH COMBINED
df_pmd_22_23 = df_pmd_22_23.iloc[3:]
df_pmd_22_23 = df_pmd_22_23.reset_index(drop=True)

#SAVE TO CSV (PROCESSED DATA FOLDER)
df_pmd_22_23.to_csv("df_pmd_22_23.csv")

In [23]:
#---------2022-2023 MORE CLEANING: REPLACING * , DROPPING LAST ROW, CHANGING DATATYPES, DROPPING % COLUMNS---------#

#FOR NULL COUNTY ROWS 

#1. Replace asterisks with NaN
import numpy as np
df_pmd_22_23=df_pmd_22_23.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

#2. Drop last row, which just has "small sample size" text
df_pmd_22_23 = df_pmd_22_23.iloc[:-1]

#3. NEW STEP 
df_pmd_22_23.iloc[4, 1:10] = np.nan
df_pmd_22_23.iloc[10, 1:10] = np.nan
df_pmd_22_23.iloc[22, 1:10] = np.nan
df_pmd_22_23.iloc[27, 1:10] = np.nan
df_pmd_22_23.iloc[30, 1:10] = np.nan

#4. Change all datatypes (except those in county) to floats 
#found this on stack overflow to change datatypes of all but one column 

ignore = ['County']

#NEED TO SAVE THIS TO THE CURRENT DATAFRAME WITH DF=
df_pmd_22_23 = (df_pmd_22_23.set_index(ignore, append=True)
        .astype(float)
        .reset_index(ignore)
       )

# df_pmd_24_25.info()

#https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

#4.Drop percentage columns and then recalculate later 
df_pmd_22_23=df_pmd_22_23.drop(["Antlered_% Change", "Antlerless_% Change", "Total_% Change"], axis=1)
df_pmd_22_23

,County,Antlered_2021-22,Antlered_2022-23,Antlerless_2021-22,Antlerless_2022-23,Total_2021-22,Total_2022-23
0,Allegany,1837.0,1925.0,1177.0,1474.0,3014.0,3399.0
1,Anne Arundel,755.0,800.0,1082.0,1083.0,1837.0,1883.0
2,Baltimore,1680.0,1856.0,2821.0,2990.0,4501.0,4846.0
3,Calvert,551.0,599.0,787.0,879.0,1338.0,1478.0
4,Caroline,NaN,NaN,NaN,NaN,NaN,NaN
5,whitetail,908.0,933.0,1576.0,1967.0,2484.0,2900.0
6,sika,0.0,0.0,0.0,1.0,0.0,1.0
7,Carroll,2058.0,2351.0,3013.0,3434.0,5071.0,5785.0
8,Cecil,1197.0,1230.0,1996.0,2122.0,3193.0,3352.0
9,Charles,873.0,1136.0,1082.0,1439.0,1955.0,2575.0


In [24]:
#---------2022-2023 MORE CLEANING: TEST TRIAL FOR COMBINING SUMS---------#
#5. Add whitetail and sika counts for each county manually, and then fill those sums into currently NaN county row 

#CAROLINE COUNTY
df_pmd_22_23.iloc[4, 1] = df_pmd_22_23.iloc[[5, 6], 1].sum()
df_pmd_22_23.iloc[4, 2] = df_pmd_22_23.iloc[[5,6], 2].sum()
df_pmd_22_23.iloc[4, 3] = df_pmd_22_23.iloc[[5, 6], 3].sum()
df_pmd_22_23.iloc[4, 4] = df_pmd_22_23.iloc[[5,6], 4].sum()
df_pmd_22_23.iloc[4, 5] = df_pmd_22_23.iloc[[5, 6], 5].sum()
df_pmd_22_23.iloc[4, 6] = df_pmd_22_23.iloc[[5,6], 6].sum()

df_pmd_22_23

,County,Antlered_2021-22,Antlered_2022-23,Antlerless_2021-22,Antlerless_2022-23,Total_2021-22,Total_2022-23
0,Allegany,1837.0,1925.0,1177.0,1474.0,3014.0,3399.0
1,Anne Arundel,755.0,800.0,1082.0,1083.0,1837.0,1883.0
2,Baltimore,1680.0,1856.0,2821.0,2990.0,4501.0,4846.0
3,Calvert,551.0,599.0,787.0,879.0,1338.0,1478.0
4,Caroline,908.0,933.0,1576.0,1968.0,2484.0,2901.0
5,whitetail,908.0,933.0,1576.0,1967.0,2484.0,2900.0
6,sika,0.0,0.0,0.0,1.0,0.0,1.0
7,Carroll,2058.0,2351.0,3013.0,3434.0,5071.0,5785.0
8,Cecil,1197.0,1230.0,1996.0,2122.0,3193.0,3352.0
9,Charles,873.0,1136.0,1082.0,1439.0,1955.0,2575.0


In [25]:
#---------2023-2024 MORE CLEANING: COMBINING ALL SUMS FOR WHITETAIL AND SIKA FOR ALL APPLICABLE COUNTIES---------#

# Dorchester
df_pmd_22_23.iloc[10, 1] = df_pmd_22_23.iloc[[11, 12], 1].sum()
df_pmd_22_23.iloc[10, 2] = df_pmd_22_23.iloc[[11,12], 2].sum()
df_pmd_22_23.iloc[10, 3] = df_pmd_22_23.iloc[[11, 12], 3].sum()
df_pmd_22_23.iloc[10, 4] = df_pmd_22_23.iloc[[11,12], 4].sum()
df_pmd_22_23.iloc[10, 5] = df_pmd_22_23.iloc[[11, 12], 5].sum()
df_pmd_22_23.iloc[10, 6] = df_pmd_22_23.iloc[[11,12], 6].sum()

# Somerset
df_pmd_22_23.iloc[22, 1] = df_pmd_22_23.iloc[[23, 24], 1].sum()
df_pmd_22_23.iloc[22, 2] = df_pmd_22_23.iloc[[23,24], 2].sum()
df_pmd_22_23.iloc[22, 3] = df_pmd_22_23.iloc[[23, 24], 3].sum()
df_pmd_22_23.iloc[22, 4] = df_pmd_22_23.iloc[[23,24], 4].sum()
df_pmd_22_23.iloc[22, 5] = df_pmd_22_23.iloc[[23, 24], 5].sum()
df_pmd_22_23.iloc[22, 6] = df_pmd_22_23.iloc[[23,24], 6].sum()

# Wicomico
df_pmd_22_23.iloc[27, 1] = df_pmd_22_23.iloc[[28, 29], 1].sum()
df_pmd_22_23.iloc[27, 2] = df_pmd_22_23.iloc[[28,29], 2].sum()
df_pmd_22_23.iloc[27, 3] = df_pmd_22_23.iloc[[28, 29], 3].sum()
df_pmd_22_23.iloc[27, 4] = df_pmd_22_23.iloc[[28,29], 4].sum()
df_pmd_22_23.iloc[27, 5] = df_pmd_22_23.iloc[[28, 29], 5].sum()
df_pmd_22_23.iloc[27, 6] = df_pmd_22_23.iloc[[28,29], 6].sum()

# Worcester
df_pmd_22_23.iloc[30, 1] = df_pmd_22_23.iloc[[31, 32], 1].sum()
df_pmd_22_23.iloc[30, 2] = df_pmd_22_23.iloc[[31,32], 2].sum()
df_pmd_22_23.iloc[30, 3] = df_pmd_22_23.iloc[[31, 32], 3].sum()
df_pmd_22_23.iloc[30, 4] = df_pmd_22_23.iloc[[31,32], 4].sum()
df_pmd_22_23.iloc[30, 5] = df_pmd_22_23.iloc[[31, 32], 5].sum()
df_pmd_22_23.iloc[30, 6] = df_pmd_22_23.iloc[[31,32], 6].sum()

df_pmd_22_23

,County,Antlered_2021-22,Antlered_2022-23,Antlerless_2021-22,Antlerless_2022-23,Total_2021-22,Total_2022-23
0,Allegany,1837.0,1925.0,1177.0,1474.0,3014.0,3399.0
1,Anne Arundel,755.0,800.0,1082.0,1083.0,1837.0,1883.0
2,Baltimore,1680.0,1856.0,2821.0,2990.0,4501.0,4846.0
3,Calvert,551.0,599.0,787.0,879.0,1338.0,1478.0
4,Caroline,908.0,933.0,1576.0,1968.0,2484.0,2901.0
5,whitetail,908.0,933.0,1576.0,1967.0,2484.0,2900.0
6,sika,0.0,0.0,0.0,1.0,0.0,1.0
7,Carroll,2058.0,2351.0,3013.0,3434.0,5071.0,5785.0
8,Cecil,1197.0,1230.0,1996.0,2122.0,3193.0,3352.0
9,Charles,873.0,1136.0,1082.0,1439.0,1955.0,2575.0


In [26]:
#---------2022-2023 FINAL CLEANING: DROPPING ROWS---------#
#Filter county values for "whitetail" and "sika" and drop those rows
df_pmd_22_23=df_pmd_22_23[ df_pmd_22_23[ "County" ].str.contains( "whitetail|sika" )== False]
df_pmd_22_23 = df_pmd_22_23.reset_index(drop=True)
df_pmd_22_23.to_csv("df_pmd_22_23.csv")

In [27]:
#2021-2022 Processed Harvest Data Pt. 1

#PANDAS
df_pmd_21_22 = pd.read_html("https://news.maryland.gov/dnr/2022/02/10/maryland-hunters-report-taking-70845-deer-in-2021-2022-season/", storage_options={"User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:52.0) Gecko/20100101 Firefox/52.0"})
df_pmd_21_22 = df_pmd_21_22[0]
#type(df_pmd_21_22)
df_pmd_21_22.to_csv("df_pmd_21_22.csv")

In [28]:
#2021-2022 Processed Harvest Data pt. 2 = COMBINING HEADERS

header_row_one_2122 = df_pmd_21_22.iloc[1]
header_row_two_2122 = df_pmd_21_22.iloc[2]

new_columns = []

for header_one, header_two in zip(header_row_one_2122, header_row_two_2122):
    if pd.isna(header_one):
    #https://pandas.pydata.org/docs/reference/api/pandas.isna.html 
        combined_header = header_two 
    else:
        combined_header = f"{header_one}_{header_two}"
    new_columns.append(combined_header)

#STORE NEW COMBINED HEADERS
df_pmd_21_22.columns = new_columns

#REPLACE OLD HEADERS WITH COMBINED
df_pmd_21_22 = df_pmd_21_22.iloc[3:]
df_pmd_21_22 = df_pmd_21_22.reset_index(drop=True)

df_pmd_21_22

,County,Antlered_2021-22,Antlered_2020-21,Antlered_% Change,Antlerless_2021-22,Antlerless_2020-21,Antlerless_% Change,Total_2021-22,Total_2020-21,Total_% Change
0,Allegany,1837,1921,-4.4,1177,1379,-14.6,3014,3300,-8.7
1,Anne Arundel,755,642,17.6,1082,1277,-15.3,1837,1919,-4.3
2,Baltimore,1680,1781,-5.7,2821,3575,-21.1,4501,5356,-16.0
3,Calvert,551,468,17.7,787,916,-14.1,1338,1384,-3.3
4,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline
5,whitetail,908,884,2.7,1576,1894,-16.8,2484,2778,-10.6
6,sika,0,0,*,0,2,*,0,2,*
7,Carroll,2058,2196,-6.3,3013,4006,-24.8,5071,6202,-18.2
8,Cecil,1197,1227,-2.4,1996,2530,-21.1,3193,3757,-15.0
9,Charles,873,1074,-18.7,1082,1557,-30.5,1955,2631,-25.7


In [29]:
#---------2021-2022 MORE CLEANING: REPLACING * , DROPPING LAST ROW, CHANGING DATATYPES, DROPPING % COLUMNS---------#

#FOR FILLED COUNTY ROWS 

#1. Replace asterisks with NaN
import numpy as np
df_pmd_21_22=df_pmd_21_22.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

#2. Drop last row, which just has "small sample size" text
df_pmd_21_22 = df_pmd_21_22.iloc[:-1]

#3. NEW STEP 
df_pmd_21_22.iloc[4, 1:10] = np.nan
df_pmd_21_22.iloc[10, 1:10] = np.nan
df_pmd_21_22.iloc[22, 1:10] = np.nan
df_pmd_21_22.iloc[27, 1:10] = np.nan
df_pmd_21_22.iloc[30, 1:10] = np.nan

#4. Change all datatypes (except those in county) to floats 
#found this on stack overflow to change datatypes of all but one column 

ignore = ['County']

# #NEED TO SAVE THIS TO THE CURRENT DATAFRAME WITH DF=
df_pmd_21_22 = (df_pmd_21_22.set_index(ignore, append=True)
        .astype(float)
        .reset_index(ignore)
       )

# # df_pmd_21_22.info()

# #https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

# #4.Drop percentage columns and then recalculate later 
df_pmd_21_22=df_pmd_21_22.drop(["Antlered_% Change", "Antlerless_% Change", "Total_% Change"], axis=1)
df_pmd_21_22

,County,Antlered_2021-22,Antlered_2020-21,Antlerless_2021-22,Antlerless_2020-21,Total_2021-22,Total_2020-21
0,Allegany,1837.0,1921.0,1177.0,1379.0,3014.000,3300.0
1,Anne Arundel,755.0,642.0,1082.0,1277.0,1837.000,1919.0
2,Baltimore,1680.0,1781.0,2821.0,3575.0,4501.000,5356.0
3,Calvert,551.0,468.0,787.0,916.0,1338.000,1384.0
4,Caroline,NaN,NaN,NaN,NaN,NaN,NaN
5,whitetail,908.0,884.0,1576.0,1894.0,2484.000,2778.0
6,sika,0.0,0.0,0.0,2.0,0.000,2.0
7,Carroll,2058.0,2196.0,3013.0,4006.0,5071.000,6202.0
8,Cecil,1197.0,1227.0,1996.0,2530.0,3193.000,3757.0
9,Charles,873.0,1074.0,1082.0,1557.0,1955.000,2631.0


In [30]:
#---------2021-2022 MORE CLEANING: TEST TRIAL FOR COMBINING SUMS---------#
#5. Add whitetail and sika counts for each county manually, and then fill those sums into currently NaN county row 

#CAROLINE COUNTY
df_pmd_21_22.iloc[4, 1] = df_pmd_21_22.iloc[[5, 6], 1].sum()
df_pmd_21_22.iloc[4, 2] = df_pmd_21_22.iloc[[5,6], 2].sum()
df_pmd_21_22.iloc[4, 3] = df_pmd_21_22.iloc[[5, 6], 3].sum()
df_pmd_21_22.iloc[4, 4] = df_pmd_21_22.iloc[[5,6], 4].sum()
df_pmd_21_22.iloc[4, 5] = df_pmd_21_22.iloc[[5, 6], 5].sum()
df_pmd_21_22.iloc[4, 6] = df_pmd_21_22.iloc[[5,6], 6].sum()

df_pmd_21_22

,County,Antlered_2021-22,Antlered_2020-21,Antlerless_2021-22,Antlerless_2020-21,Total_2021-22,Total_2020-21
0,Allegany,1837.0,1921.0,1177.0,1379.0,3014.000,3300.0
1,Anne Arundel,755.0,642.0,1082.0,1277.0,1837.000,1919.0
2,Baltimore,1680.0,1781.0,2821.0,3575.0,4501.000,5356.0
3,Calvert,551.0,468.0,787.0,916.0,1338.000,1384.0
4,Caroline,908.0,884.0,1576.0,1896.0,2484.000,2780.0
5,whitetail,908.0,884.0,1576.0,1894.0,2484.000,2778.0
6,sika,0.0,0.0,0.0,2.0,0.000,2.0
7,Carroll,2058.0,2196.0,3013.0,4006.0,5071.000,6202.0
8,Cecil,1197.0,1227.0,1996.0,2530.0,3193.000,3757.0
9,Charles,873.0,1074.0,1082.0,1557.0,1955.000,2631.0


In [31]:
#---------2021-2022 MORE CLEANING: COMBINING ALL SUMS FOR WHITETAIL AND SIKA FOR ALL APPLICABLE COUNTIES---------#

# #Dorchester
df_pmd_21_22.iloc[10, 1] = df_pmd_21_22.iloc[[11, 12], 1].sum()
df_pmd_21_22.iloc[10, 2] = df_pmd_21_22.iloc[[11,12], 2].sum()
df_pmd_21_22.iloc[10, 3] = df_pmd_21_22.iloc[[11, 12], 3].sum()
df_pmd_21_22.iloc[10, 4] = df_pmd_21_22.iloc[[11,12], 4].sum()
df_pmd_21_22.iloc[10, 5] = df_pmd_21_22.iloc[[11, 12], 5].sum()
df_pmd_21_22.iloc[10, 6] = df_pmd_21_22.iloc[[11,12], 6].sum()

# #Somerset
df_pmd_21_22.iloc[22, 1] = df_pmd_21_22.iloc[[23, 24], 1].sum()
df_pmd_21_22.iloc[22, 2] = df_pmd_21_22.iloc[[23,24], 2].sum()
df_pmd_21_22.iloc[22, 3] = df_pmd_21_22.iloc[[23, 24], 3].sum()
df_pmd_21_22.iloc[22, 4] = df_pmd_21_22.iloc[[23,24], 4].sum()
df_pmd_21_22.iloc[22, 5] = df_pmd_21_22.iloc[[23, 24], 5].sum()
df_pmd_21_22.iloc[22, 6] = df_pmd_21_22.iloc[[23,24], 6].sum()

# # #Wicomico
df_pmd_21_22.iloc[27, 1] = df_pmd_21_22.iloc[[28, 29], 1].sum()
df_pmd_21_22.iloc[27, 2] = df_pmd_21_22.iloc[[28,29], 2].sum()
df_pmd_21_22.iloc[27, 3] = df_pmd_21_22.iloc[[28, 29], 3].sum()
df_pmd_21_22.iloc[27, 4] = df_pmd_21_22.iloc[[28,29], 4].sum()
df_pmd_21_22.iloc[27, 5] = df_pmd_21_22.iloc[[28, 29], 5].sum()
df_pmd_21_22.iloc[27, 6] = df_pmd_21_22.iloc[[28,29], 6].sum()

# # #Worcester
df_pmd_21_22.iloc[30, 1] = df_pmd_21_22.iloc[[31, 32], 1].sum()
df_pmd_21_22.iloc[30, 2] = df_pmd_21_22.iloc[[31,32], 2].sum()
df_pmd_21_22.iloc[30, 3] = df_pmd_21_22.iloc[[31, 32], 3].sum()
df_pmd_21_22.iloc[30, 4] = df_pmd_21_22.iloc[[31,32], 4].sum()
df_pmd_21_22.iloc[30, 5] = df_pmd_21_22.iloc[[31, 32], 5].sum()
df_pmd_21_22.iloc[30, 6] = df_pmd_21_22.iloc[[31,32], 6].sum()

df_pmd_21_22

,County,Antlered_2021-22,Antlered_2020-21,Antlerless_2021-22,Antlerless_2020-21,Total_2021-22,Total_2020-21
0,Allegany,1837.0,1921.0,1177.0,1379.0,3014.000,3300.0
1,Anne Arundel,755.0,642.0,1082.0,1277.0,1837.000,1919.0
2,Baltimore,1680.0,1781.0,2821.0,3575.0,4501.000,5356.0
3,Calvert,551.0,468.0,787.0,916.0,1338.000,1384.0
4,Caroline,908.0,884.0,1576.0,1896.0,2484.000,2780.0
5,whitetail,908.0,884.0,1576.0,1894.0,2484.000,2778.0
6,sika,0.0,0.0,0.0,2.0,0.000,2.0
7,Carroll,2058.0,2196.0,3013.0,4006.0,5071.000,6202.0
8,Cecil,1197.0,1227.0,1996.0,2530.0,3193.000,3757.0
9,Charles,873.0,1074.0,1082.0,1557.0,1955.000,2631.0


In [32]:
#---------2021-2022 FINAL CLEANING: DROPPING ROWS---------#
#Filter county values for "whitetail" and "sika" and drop those rows
df_pmd_21_22=df_pmd_21_22[ df_pmd_21_22[ "County" ].str.contains( "whitetail|sika" )== False]
df_pmd_22_23 = df_pmd_22_23.reset_index(drop=True)
df_pmd_21_22.to_csv("df_pmd_21_22.csv")

In [33]:
#2020-2021 Processed Harvest Data Pt. 1

#PANDAS
df_pmd_20_21 = pd.read_html("https://news.maryland.gov/dnr/2021/02/16/maryland-hunters-harvest-81000-deer-during-2020-2021-season/", storage_options={"User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:52.0) Gecko/20100101 Firefox/52.0"})
df_pmd_20_21 = df_pmd_20_21[0]
#type(df_pmd_20_21)
df_pmd_20_21.to_csv("df_pmd_20_21.csv")

In [34]:
#2020-2021 Processed Harvest Data pt. 2 = COMBINING HEADERS

#COMBINE HEADERS 
header_row_one_2021 = df_pmd_20_21.iloc[1]
header_row_two_2021 = df_pmd_20_21.iloc[2]

new_columns = []

for header_one, header_two in zip(header_row_one_2021, header_row_two_2021):
    if pd.isna(header_one):
    #https://pandas.pydata.org/docs/reference/api/pandas.isna.html 
        combined_header = header_two 
    else:
        combined_header = f"{header_one}_{header_two}"
    new_columns.append(combined_header)

#STORE NEW COMBINED HEADERS
df_pmd_20_21.columns = new_columns

#REPLACE OLD HEADERS WITH COMBINED
df_pmd_20_21 = df_pmd_20_21.iloc[3:]
df_pmd_20_21 = df_pmd_20_21.reset_index(drop=True)

df_pmd_20_21

,County,Antlered_2019-20,Antlered_2020-21,Antlered_% Change,Antlerless_2019-20,Antlerless_2020-21,Antlerless_% Change,Total_2019-20,Total_2020-21,Total_% Change
0,Allegany,1667,1921,15.2,1179,1379,17.0,2846,3300,16.0
1,Anne Arundel,966,642,-33.5,1653,1277,-22.7,2619,1919,-26.7
2,Baltimore,1774,1781,0.4,3195,3575,11.9,4969,5356,7.8
3,Calvert,639,468,-26.8,1025,916,-10.6,1664,1384,-16.8
4,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline
5,whitetail,785,884,12.6,1797,1894,5.4,2582,2778,7.6
6,sika,0,0,*,1,2,*,1,2,*
7,Carroll,2016,2196,8.9,3537,4006,13.3,5553,6202,11.7
8,Cecil,1226,1227,0.1,2468,2530,2.5,3694,3757,1.7
9,Charles,1349,1074,-20.4,1868,1557,-16.6,3217,2631,-18.2


In [35]:
#---------2020-2021 MORE CLEANING: REPLACING * , DROPPING LAST ROW, CHANGING DATATYPES, DROPPING % COLUMNS---------#

#FOR FILLED COUNTY ROWS 

#1. Replace asterisks with NaN
import numpy as np
df_pmd_20_21=df_pmd_20_21.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

#2. Drop last row, which just has "small sample size" text
df_pmd_20_21 = df_pmd_20_21.iloc[:-1]

#3. NEW STEP 
df_pmd_20_21.iloc[4, 1:10] = np.nan
df_pmd_20_21.iloc[10, 1:10] = np.nan
df_pmd_20_21.iloc[22, 1:10] = np.nan
df_pmd_20_21.iloc[27, 1:10] = np.nan
df_pmd_20_21.iloc[30, 1:10] = np.nan

#4. Change all datatypes (except those in county) to floats 
#found this on stack overflow to change datatypes of all but one column 

ignore = ['County']

#NEED TO SAVE THIS TO THE CURRENT DATAFRAME WITH DF=
df_pmd_20_21 = (df_pmd_20_21.set_index(ignore, append=True)
        .astype(float)
        .reset_index(ignore)
       )

# df_pmd_20_21.info()

#https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

#4.Drop percentage columns and then recalculate later 
df_pmd_20_21=df_pmd_20_21.drop(["Antlered_% Change", "Antlerless_% Change", "Total_% Change"], axis=1)
df_pmd_20_21

,County,Antlered_2019-20,Antlered_2020-21,Antlerless_2019-20,Antlerless_2020-21,Total_2019-20,Total_2020-21
0,Allegany,1667.0,1921.0,1179.0,1379.0,2846.0,3300.0
1,Anne Arundel,966.0,642.0,1653.0,1277.0,2619.0,1919.0
2,Baltimore,1774.0,1781.0,3195.0,3575.0,4969.0,5356.0
3,Calvert,639.0,468.0,1025.0,916.0,1664.0,1384.0
4,Caroline,NaN,NaN,NaN,NaN,NaN,NaN
5,whitetail,785.0,884.0,1797.0,1894.0,2582.0,2778.0
6,sika,0.0,0.0,1.0,2.0,1.0,2.0
7,Carroll,2016.0,2196.0,3537.0,4006.0,5553.0,6202.0
8,Cecil,1226.0,1227.0,2468.0,2530.0,3694.0,3757.0
9,Charles,1349.0,1074.0,1868.0,1557.0,3217.0,2631.0


In [36]:
#---------2020-2021 MORE CLEANING: TEST TRIAL FOR COMBINING SUMS---------#
#5. Add whitetail and sika counts for each county manually, and then fill those sums into currently NaN county row 

#CAROLINE COUNTY
df_pmd_20_21.iloc[4, 1] = df_pmd_20_21.iloc[[5, 6], 1].sum()
df_pmd_20_21.iloc[4, 2] = df_pmd_20_21.iloc[[5,6], 2].sum()
df_pmd_20_21.iloc[4, 3] = df_pmd_20_21.iloc[[5, 6], 3].sum()
df_pmd_20_21.iloc[4, 4] = df_pmd_20_21.iloc[[5,6], 4].sum()
df_pmd_20_21.iloc[4, 5] = df_pmd_20_21.iloc[[5, 6], 5].sum()
df_pmd_20_21.iloc[4, 6] = df_pmd_20_21.iloc[[5,6], 6].sum()

df_pmd_20_21

,County,Antlered_2019-20,Antlered_2020-21,Antlerless_2019-20,Antlerless_2020-21,Total_2019-20,Total_2020-21
0,Allegany,1667.0,1921.0,1179.0,1379.0,2846.0,3300.0
1,Anne Arundel,966.0,642.0,1653.0,1277.0,2619.0,1919.0
2,Baltimore,1774.0,1781.0,3195.0,3575.0,4969.0,5356.0
3,Calvert,639.0,468.0,1025.0,916.0,1664.0,1384.0
4,Caroline,785.0,884.0,1798.0,1896.0,2583.0,2780.0
5,whitetail,785.0,884.0,1797.0,1894.0,2582.0,2778.0
6,sika,0.0,0.0,1.0,2.0,1.0,2.0
7,Carroll,2016.0,2196.0,3537.0,4006.0,5553.0,6202.0
8,Cecil,1226.0,1227.0,2468.0,2530.0,3694.0,3757.0
9,Charles,1349.0,1074.0,1868.0,1557.0,3217.0,2631.0


In [37]:
#---------2020-2021 MORE CLEANING: COMBINING ALL SUMS FOR WHITETAIL AND SIKA FOR ALL APPLICABLE COUNTIES---------#

# #Dorchester
df_pmd_20_21.iloc[10, 1] = df_pmd_20_21.iloc[[11, 12], 1].sum()
df_pmd_20_21.iloc[10, 2] = df_pmd_20_21.iloc[[11,12], 2].sum()
df_pmd_20_21.iloc[10, 3] = df_pmd_20_21.iloc[[11, 12], 3].sum()
df_pmd_20_21.iloc[10, 4] = df_pmd_20_21.iloc[[11,12], 4].sum()
df_pmd_20_21.iloc[10, 5] = df_pmd_20_21.iloc[[11, 12], 5].sum()
df_pmd_20_21.iloc[10, 6] = df_pmd_20_21.iloc[[11,12], 6].sum()

# #Somerset
df_pmd_20_21.iloc[22, 1] = df_pmd_20_21.iloc[[23, 24], 1].sum()
df_pmd_20_21.iloc[22, 2] = df_pmd_20_21.iloc[[23,24], 2].sum()
df_pmd_20_21.iloc[22, 3] = df_pmd_20_21.iloc[[23, 24], 3].sum()
df_pmd_20_21.iloc[22, 4] = df_pmd_20_21.iloc[[23,24], 4].sum()
df_pmd_20_21.iloc[22, 5] = df_pmd_20_21.iloc[[23, 24], 5].sum()
df_pmd_20_21.iloc[22, 6] = df_pmd_20_21.iloc[[23,24], 6].sum()

# # #Wicomico
df_pmd_20_21.iloc[27, 1] = df_pmd_20_21.iloc[[28, 29], 1].sum()
df_pmd_20_21.iloc[27, 2] = df_pmd_20_21.iloc[[28,29], 2].sum()
df_pmd_20_21.iloc[27, 3] = df_pmd_20_21.iloc[[28, 29], 3].sum()
df_pmd_20_21.iloc[27, 4] = df_pmd_20_21.iloc[[28,29], 4].sum()
df_pmd_20_21.iloc[27, 5] = df_pmd_20_21.iloc[[28, 29], 5].sum()
df_pmd_20_21.iloc[27, 6] = df_pmd_20_21.iloc[[28,29], 6].sum()

# # #Worcester
df_pmd_20_21.iloc[30, 1] = df_pmd_20_21.iloc[[31, 32], 1].sum()
df_pmd_20_21.iloc[30, 2] = df_pmd_20_21.iloc[[31,32], 2].sum()
df_pmd_20_21.iloc[30, 3] = df_pmd_20_21.iloc[[31, 32], 3].sum()
df_pmd_20_21.iloc[30, 4] = df_pmd_20_21.iloc[[31,32], 4].sum()
df_pmd_20_21.iloc[30, 5] = df_pmd_20_21.iloc[[31, 32], 5].sum()
df_pmd_20_21.iloc[30, 6] = df_pmd_20_21.iloc[[31,32], 6].sum()

df_pmd_20_21

,County,Antlered_2019-20,Antlered_2020-21,Antlerless_2019-20,Antlerless_2020-21,Total_2019-20,Total_2020-21
0,Allegany,1667.0,1921.0,1179.0,1379.0,2846.0,3300.0
1,Anne Arundel,966.0,642.0,1653.0,1277.0,2619.0,1919.0
2,Baltimore,1774.0,1781.0,3195.0,3575.0,4969.0,5356.0
3,Calvert,639.0,468.0,1025.0,916.0,1664.0,1384.0
4,Caroline,785.0,884.0,1798.0,1896.0,2583.0,2780.0
5,whitetail,785.0,884.0,1797.0,1894.0,2582.0,2778.0
6,sika,0.0,0.0,1.0,2.0,1.0,2.0
7,Carroll,2016.0,2196.0,3537.0,4006.0,5553.0,6202.0
8,Cecil,1226.0,1227.0,2468.0,2530.0,3694.0,3757.0
9,Charles,1349.0,1074.0,1868.0,1557.0,3217.0,2631.0


In [38]:
#---------2020-2021 FINAL CLEANING: DROPPING ROWS---------#
#Filter county values for "whitetail" and "sika" and drop those rows
df_pmd_20_21=df_pmd_20_21[ df_pmd_20_21[ "County" ].str.contains( "whitetail|sika" )== False]
df_pmd_20_21 = df_pmd_20_21.reset_index(drop=True)
df_pmd_20_21.to_csv("df_pmd_20_21.csv")

In [39]:
#2019-2020 Processed Harvest Data Pt. 1

#PANDAS
df_pmd_19_20 = pd.read_html("https://news.maryland.gov/dnr/2020/02/13/maryland-hunters-harvest-nearly-80000-deer-during-2019-2020-season/", storage_options={"User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:52.0) Gecko/20100101 Firefox/52.0"})
df_pmd_19_20 = df_pmd_19_20[0]
#type(df_pmd_19_20)
df_pmd_19_20.to_csv("df_pmd_19_20.csv")

In [40]:
#2019-2020 Processed Harvest Data pt. 2 = COMBINING HEADERS 

header_row_one_1920 = df_pmd_19_20.iloc[1]
header_row_two_1920 = df_pmd_19_20.iloc[2]

new_columns = []

for header_one, header_two in zip(header_row_one_1920, header_row_two_1920):
    if pd.isna(header_one):
    #https://pandas.pydata.org/docs/reference/api/pandas.isna.html 
        combined_header = header_two 
    else:
        combined_header = f"{header_one}_{header_two}"
    new_columns.append(combined_header)

#STORE NEW COMBINED HEADERS
df_pmd_19_20.columns = new_columns

#REPLACE OLD HEADERS WITH COMBINED
df_pmd_19_20 = df_pmd_19_20.iloc[3:]
df_pmd_19_20 = df_pmd_19_20.reset_index(drop=True)

df_pmd_19_20

,County,Antlered_2019-20,Antlered_2018-19,Antlered_% Change,Antlerless_2019-20,Antlerless_2018-19,Antlerless_% Change,Total_2019-20,Total_2018-19,Total_% Change
0,Allegany,1667,2017,-17.4,1179,1444,-18.4,2846,3461,-17.8
1,Anne Arundel,966,920,5.0,1653,1562,5.8,2619,2482,5.5
2,Baltimore,1774,1641,8.1,3195,2963,7.8,4969,4604,7.9
3,Calvert,639,614,4.1,1025,947,8.2,1664,1561,6.6
4,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline
5,whitetail,785,829,-5.3,1797,1727,4.1,2582,2556,1.0
6,sika,0,0,*,1,1,*,1,1,*
7,Carroll,2016,2197,-8.2,3537,3224,9.7,5553,5421,2.4
8,Cecil,Cecil,Cecil,Cecil,Cecil,Cecil,Cecil,Cecil,Cecil,Cecil
9,whitetail,1226,1237,-0.9,2468,2269,8.8,3694,3506,5.4


In [41]:
#---------2019-2020 MORE CLEANING: REPLACING * , DROPPING LAST ROW, CHANGING DATATYPES, DROPPING % COLUMNS---------#

#FOR FILLED COUNTY ROWS 

#1. Replace asterisks with NaN
import numpy as np
df_pmd_19_20=df_pmd_19_20.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

#2. Drop last row, which just has "small sample size" text
df_pmd_19_20 = df_pmd_19_20.iloc[:-1]

#3. NEW STEP 
df_pmd_19_20.iloc[4, 1:10] = np.nan
df_pmd_19_20.iloc[8, 1:10] = np.nan
df_pmd_19_20.iloc[12, 1:10] = np.nan
df_pmd_19_20.iloc[24, 1:10] = np.nan
df_pmd_19_20.iloc[29, 1:10] = np.nan
df_pmd_19_20.iloc[32, 1:10] = np.nan

#4. Change all datatypes (except those in county) to floats 
#found this on stack overflow to change datatypes of all but one column 

ignore = ['County']

#NEED TO SAVE THIS TO THE CURRENT DATAFRAME WITH DF=
df_pmd_19_20 = (df_pmd_19_20.set_index(ignore, append=True)
        .astype(float)
        .reset_index(ignore)
       )

# df_pmd_19_20.info()

#https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

#4.Drop percentage columns and then recalculate later 
df_pmd_19_20=df_pmd_19_20.drop(["Antlered_% Change", "Antlerless_% Change", "Total_% Change"], axis=1)
df_pmd_19_20

,County,Antlered_2019-20,Antlered_2018-19,Antlerless_2019-20,Antlerless_2018-19,Total_2019-20,Total_2018-19
0,Allegany,1667.0,2017.0,1179.0,1444.0,2846.0,3461.0
1,Anne Arundel,966.0,920.0,1653.0,1562.0,2619.0,2482.0
2,Baltimore,1774.0,1641.0,3195.0,2963.0,4969.0,4604.0
3,Calvert,639.0,614.0,1025.0,947.0,1664.0,1561.0
4,Caroline,NaN,NaN,NaN,NaN,NaN,NaN
5,whitetail,785.0,829.0,1797.0,1727.0,2582.0,2556.0
6,sika,0.0,0.0,1.0,1.0,1.0,1.0
7,Carroll,2016.0,2197.0,3537.0,3224.0,5553.0,5421.0
8,Cecil,NaN,NaN,NaN,NaN,NaN,NaN
9,whitetail,1226.0,1237.0,2468.0,2269.0,3694.0,3506.0


In [42]:
#---------2019-2020 MORE CLEANING: TEST TRIAL FOR COMBINING SUMS---------#
#5. Add whitetail and sika counts for each county manually, and then fill those sums into currently NaN county row 

#CAROLINE COUNTY
df_pmd_19_20.iloc[4, 1] = df_pmd_19_20.iloc[[5, 6], 1].sum()
df_pmd_19_20.iloc[4, 2] = df_pmd_19_20.iloc[[5,6], 2].sum()
df_pmd_19_20.iloc[4, 3] = df_pmd_19_20.iloc[[5, 6], 3].sum()
df_pmd_19_20.iloc[4, 4] = df_pmd_19_20.iloc[[5,6], 4].sum()
df_pmd_19_20.iloc[4, 5] = df_pmd_19_20.iloc[[5, 6], 5].sum()
df_pmd_19_20.iloc[4, 6] = df_pmd_19_20.iloc[[5,6], 6].sum()

df_pmd_19_20

,County,Antlered_2019-20,Antlered_2018-19,Antlerless_2019-20,Antlerless_2018-19,Total_2019-20,Total_2018-19
0,Allegany,1667.0,2017.0,1179.0,1444.0,2846.0,3461.0
1,Anne Arundel,966.0,920.0,1653.0,1562.0,2619.0,2482.0
2,Baltimore,1774.0,1641.0,3195.0,2963.0,4969.0,4604.0
3,Calvert,639.0,614.0,1025.0,947.0,1664.0,1561.0
4,Caroline,785.0,829.0,1798.0,1728.0,2583.0,2557.0
5,whitetail,785.0,829.0,1797.0,1727.0,2582.0,2556.0
6,sika,0.0,0.0,1.0,1.0,1.0,1.0
7,Carroll,2016.0,2197.0,3537.0,3224.0,5553.0,5421.0
8,Cecil,NaN,NaN,NaN,NaN,NaN,NaN
9,whitetail,1226.0,1237.0,2468.0,2269.0,3694.0,3506.0


In [43]:
#---------2019-2020 MORE CLEANING: COMBINING ALL SUMS FOR WHITETAIL AND SIKA FOR ALL APPLICABLE COUNTIES---------#

# Cecil
df_pmd_19_20.iloc[8, 1] = df_pmd_19_20.iloc[[9, 10], 1].sum()
df_pmd_19_20.iloc[8, 2] = df_pmd_19_20.iloc[[9, 10], 2].sum()
df_pmd_19_20.iloc[8, 3] = df_pmd_19_20.iloc[[9, 10], 3].sum()
df_pmd_19_20.iloc[8, 4] = df_pmd_19_20.iloc[[9, 10], 4].sum()
df_pmd_19_20.iloc[8, 5] = df_pmd_19_20.iloc[[9, 10], 5].sum()
df_pmd_19_20.iloc[8, 6] = df_pmd_19_20.iloc[[9, 10], 6].sum()

# #Dorchester
df_pmd_19_20.iloc[12, 1] = df_pmd_19_20.iloc[[13, 14], 1].sum()
df_pmd_19_20.iloc[12, 2] = df_pmd_19_20.iloc[[13,14], 2].sum()
df_pmd_19_20.iloc[12, 3] = df_pmd_19_20.iloc[[13, 14], 3].sum()
df_pmd_19_20.iloc[12, 4] = df_pmd_19_20.iloc[[13,14], 4].sum()
df_pmd_19_20.iloc[12, 5] = df_pmd_19_20.iloc[[13, 14], 5].sum()
df_pmd_19_20.iloc[12, 6] = df_pmd_19_20.iloc[[13,14], 6].sum()

# #Somerset
df_pmd_19_20.iloc[24, 1] = df_pmd_19_20.iloc[[25, 26], 1].sum()
df_pmd_19_20.iloc[24, 2] = df_pmd_19_20.iloc[[25,26], 2].sum()
df_pmd_19_20.iloc[24, 3] = df_pmd_19_20.iloc[[25, 26], 3].sum()
df_pmd_19_20.iloc[24, 4] = df_pmd_19_20.iloc[[25,26], 4].sum()
df_pmd_19_20.iloc[24, 5] = df_pmd_19_20.iloc[[25, 26], 5].sum()
df_pmd_19_20.iloc[24, 6] = df_pmd_19_20.iloc[[25,26], 6].sum()

# # #Wicomico
df_pmd_19_20.iloc[29, 1] = df_pmd_19_20.iloc[[30, 31], 1].sum()
df_pmd_19_20.iloc[29, 2] = df_pmd_19_20.iloc[[30,31], 2].sum()
df_pmd_19_20.iloc[29, 3] = df_pmd_19_20.iloc[[30, 31], 3].sum()
df_pmd_19_20.iloc[29, 4] = df_pmd_19_20.iloc[[30,31], 4].sum()
df_pmd_19_20.iloc[29, 5] = df_pmd_19_20.iloc[[30, 31], 5].sum()
df_pmd_19_20.iloc[29, 6] = df_pmd_19_20.iloc[[30,31], 6].sum()

# # #Worcester
df_pmd_19_20.iloc[32, 1] = df_pmd_19_20.iloc[[33, 34], 1].sum()
df_pmd_19_20.iloc[32, 2] = df_pmd_19_20.iloc[[33,34], 2].sum()
df_pmd_19_20.iloc[32, 3] = df_pmd_19_20.iloc[[33, 34], 3].sum()
df_pmd_19_20.iloc[32, 4] = df_pmd_19_20.iloc[[33,34], 4].sum()
df_pmd_19_20.iloc[32, 5] = df_pmd_19_20.iloc[[33, 34], 5].sum()
df_pmd_19_20.iloc[32, 6] = df_pmd_19_20.iloc[[33,34], 6].sum()

df_pmd_19_20

,County,Antlered_2019-20,Antlered_2018-19,Antlerless_2019-20,Antlerless_2018-19,Total_2019-20,Total_2018-19
0,Allegany,1667.0,2017.0,1179.0,1444.0,2846.0,3461.0
1,Anne Arundel,966.0,920.0,1653.0,1562.0,2619.0,2482.0
2,Baltimore,1774.0,1641.0,3195.0,2963.0,4969.0,4604.0
3,Calvert,639.0,614.0,1025.0,947.0,1664.0,1561.0
4,Caroline,785.0,829.0,1798.0,1728.0,2583.0,2557.0
5,whitetail,785.0,829.0,1797.0,1727.0,2582.0,2556.0
6,sika,0.0,0.0,1.0,1.0,1.0,1.0
7,Carroll,2016.0,2197.0,3537.0,3224.0,5553.0,5421.0
8,Cecil,1227.0,1237.0,2469.0,2269.0,3695.0,3506.0
9,whitetail,1226.0,1237.0,2468.0,2269.0,3694.0,3506.0


In [44]:
#---------2019-2020 FINAL CLEANING: DROPPING ROWS---------#
#Filter county values for "whitetail" and "sika" and drop those rows
df_pmd_19_20=df_pmd_19_20[ df_pmd_19_20[ "County" ].str.contains( "whitetail|sika" )== False]
df_pmd_19_20 = df_pmd_19_20.reset_index(drop=True)
df_pmd_19_20.to_csv("df_pmd_19_20.csv")

In [45]:
#2018-2019 Processed Harvest Data Pt. 1

#PANDAS
df_pmd_18_19 = pd.read_html("https://news.maryland.gov/dnr/2019/02/15/maryland-hunters-harvest-77000-deer-during-2018-2019-season/ ", storage_options={"User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:52.0) Gecko/20100101 Firefox/52.0"})
df_pmd_18_19 = df_pmd_18_19[0]
#type(df_pmd_18_19)
df_pmd_18_19.to_csv("df_pmd_18_19.csv")

In [46]:
#2018-2019 Processed Harvest Data pt. 2 = COMBINING HEADERS

header_row_one_1819 = df_pmd_18_19.iloc[1]
header_row_two_1819 = df_pmd_18_19.iloc[2]

new_columns = []

for header_one, header_two in zip(header_row_one_1819, header_row_two_1819):
    if pd.isna(header_one):
    #https://pandas.pydata.org/docs/reference/api/pandas.isna.html 
        combined_header = header_two 
    else:
        combined_header = f"{header_one}_{header_two}"
    new_columns.append(combined_header)

#STORE NEW COMBINED HEADERS
df_pmd_18_19.columns = new_columns

#REPLACE OLD HEADERS WITH COMBINED
df_pmd_18_19 = df_pmd_18_19.iloc[3:]
df_pmd_18_19 = df_pmd_18_19.reset_index(drop=True)

df_pmd_18_19

,County,Antlered_2017-18,Antlered_2018-19,Antlered_% Change,Antlerless_2017-18,Antlerless_2018-19,Antlerless_% Change,Total_2017-18,Total_2018-19,Total_% Change
0,Allegany,2078,2017,-2.9,1283,1444,12.5,3361,3461,3.0
1,Anne Arundel,978,920,-5.9,2001,1562,-21.9,2979,2482,-16.7
2,Baltimore,1754,1641,-6.4,3805,2963,-22.1,5559,4604,-17.2
3,Calvert,558,614,10.0,1130,947,-16.2,1688,1561,-7.5
4,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline,Caroline
5,whitetail,756,829,9.7,1824,1727,-5.3,2580,2556,-0.9
6,sika,1,0,*,1,1,*,2,1,*
7,Carroll,2116,2197,3.8,3780,3224,-14.7,5896,5421,-8.1
8,Cecil,1261,1237,-1.9,2442,2269,-7.1,3703,3506,-5.3
9,Charles,1166,1031,-11.6,2246,1328,-40.9,3412,2359,-30.9


In [47]:
#---------2018-2019 MORE CLEANING: REPLACING * , DROPPING LAST ROW, CHANGING DATATYPES, DROPPING % COLUMNS---------#

#FOR FILLED COUNTY ROWS 

#1. Replace asterisks with NaN
import numpy as np
df_pmd_18_19=df_pmd_18_19.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

#2. Drop last row, which just has "small sample size" text
df_pmd_18_19 = df_pmd_18_19.iloc[:-1]

#3. NEW STEP 
df_pmd_18_19.iloc[4, 1:10] = np.nan
df_pmd_18_19.iloc[10, 1:10] = np.nan
df_pmd_18_19.iloc[22, 1:10] = np.nan
df_pmd_18_19.iloc[27, 1:10] = np.nan
df_pmd_18_19.iloc[30, 1:10] = np.nan

#4. Change all datatypes (except those in county) to floats 
#found this on stack overflow to change datatypes of all but one column 

ignore = ['County']

#NEED TO SAVE THIS TO THE CURRENT DATAFRAME WITH DF=
df_pmd_18_19 = (df_pmd_18_19.set_index(ignore, append=True)
        .astype(float)
        .reset_index(ignore)
       )

# df_pmd_18_19.info()

#https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

#4.Drop percentage columns and then recalculate later 
df_pmd_18_19=df_pmd_18_19.drop(["Antlered_% Change", "Antlerless_% Change", "Total_% Change"], axis=1)
df_pmd_18_19

,County,Antlered_2017-18,Antlered_2018-19,Antlerless_2017-18,Antlerless_2018-19,Total_2017-18,Total_2018-19
0,Allegany,2078.0,2017.0,1283.0,1444.0,3361.0,3461.0
1,Anne Arundel,978.0,920.0,2001.0,1562.0,2979.0,2482.0
2,Baltimore,1754.0,1641.0,3805.0,2963.0,5559.0,4604.0
3,Calvert,558.0,614.0,1130.0,947.0,1688.0,1561.0
4,Caroline,NaN,NaN,NaN,NaN,NaN,NaN
5,whitetail,756.0,829.0,1824.0,1727.0,2580.0,2556.0
6,sika,1.0,0.0,1.0,1.0,2.0,1.0
7,Carroll,2116.0,2197.0,3780.0,3224.0,5896.0,5421.0
8,Cecil,1261.0,1237.0,2442.0,2269.0,3703.0,3506.0
9,Charles,1166.0,1031.0,2246.0,1328.0,3412.0,2359.0


In [48]:
#---------2018-2019 MORE CLEANING: TEST TRIAL FOR COMBINING SUMS---------#
#5. Add whitetail and sika counts for each county manually, and then fill those sums into currently NaN county row 

#CAROLINE COUNTY
df_pmd_18_19.iloc[4, 1] = df_pmd_18_19.iloc[[5, 6], 1].sum()
df_pmd_18_19.iloc[4, 2] = df_pmd_18_19.iloc[[5,6], 2].sum()
df_pmd_18_19.iloc[4, 3] = df_pmd_18_19.iloc[[5, 6], 3].sum()
df_pmd_18_19.iloc[4, 4] = df_pmd_18_19.iloc[[5,6], 4].sum()
df_pmd_18_19.iloc[4, 5] = df_pmd_18_19.iloc[[5, 6], 5].sum()
df_pmd_18_19.iloc[4, 6] = df_pmd_18_19.iloc[[5,6], 6].sum()

df_pmd_18_19

,County,Antlered_2017-18,Antlered_2018-19,Antlerless_2017-18,Antlerless_2018-19,Total_2017-18,Total_2018-19
0,Allegany,2078.0,2017.0,1283.0,1444.0,3361.0,3461.0
1,Anne Arundel,978.0,920.0,2001.0,1562.0,2979.0,2482.0
2,Baltimore,1754.0,1641.0,3805.0,2963.0,5559.0,4604.0
3,Calvert,558.0,614.0,1130.0,947.0,1688.0,1561.0
4,Caroline,757.0,829.0,1825.0,1728.0,2582.0,2557.0
5,whitetail,756.0,829.0,1824.0,1727.0,2580.0,2556.0
6,sika,1.0,0.0,1.0,1.0,2.0,1.0
7,Carroll,2116.0,2197.0,3780.0,3224.0,5896.0,5421.0
8,Cecil,1261.0,1237.0,2442.0,2269.0,3703.0,3506.0
9,Charles,1166.0,1031.0,2246.0,1328.0,3412.0,2359.0


In [49]:
#---------2018-2019 MORE CLEANING: COMBINING ALL SUMS FOR WHITETAIL AND SIKA FOR ALL APPLICABLE COUNTIES---------#

# #Dorchester
df_pmd_18_19.iloc[10, 1] = df_pmd_18_19.iloc[[11, 12], 1].sum()
df_pmd_18_19.iloc[10, 2] = df_pmd_18_19.iloc[[11,12], 2].sum()
df_pmd_18_19.iloc[10, 3] = df_pmd_18_19.iloc[[11, 12], 3].sum()
df_pmd_18_19.iloc[10, 4] = df_pmd_18_19.iloc[[11,12], 4].sum()
df_pmd_18_19.iloc[10, 5] = df_pmd_18_19.iloc[[11, 12], 5].sum()
df_pmd_18_19.iloc[10, 6] = df_pmd_18_19.iloc[[11,12], 6].sum()

# #Somerset
df_pmd_18_19.iloc[22, 1] = df_pmd_18_19.iloc[[23, 24], 1].sum()
df_pmd_18_19.iloc[22, 2] = df_pmd_18_19.iloc[[23,24], 2].sum()
df_pmd_18_19.iloc[22, 3] = df_pmd_18_19.iloc[[23, 24], 3].sum()
df_pmd_18_19.iloc[22, 4] = df_pmd_18_19.iloc[[23,24], 4].sum()
df_pmd_18_19.iloc[22, 5] = df_pmd_18_19.iloc[[23, 24], 5].sum()
df_pmd_18_19.iloc[22, 6] = df_pmd_18_19.iloc[[23,24], 6].sum()

# # #Wicomico
df_pmd_18_19.iloc[27, 1] = df_pmd_18_19.iloc[[28, 29], 1].sum()
df_pmd_18_19.iloc[27, 2] = df_pmd_18_19.iloc[[28,29], 2].sum()
df_pmd_18_19.iloc[27, 3] = df_pmd_18_19.iloc[[28, 29], 3].sum()
df_pmd_18_19.iloc[27, 4] = df_pmd_18_19.iloc[[28,29], 4].sum()
df_pmd_18_19.iloc[27, 5] = df_pmd_18_19.iloc[[28, 29], 5].sum()
df_pmd_18_19.iloc[27, 6] = df_pmd_18_19.iloc[[28,29], 6].sum()

# # #Worcester
df_pmd_18_19.iloc[30, 1] = df_pmd_18_19.iloc[[31, 32], 1].sum()
df_pmd_18_19.iloc[30, 2] = df_pmd_18_19.iloc[[31,32], 2].sum()
df_pmd_18_19.iloc[30, 3] = df_pmd_18_19.iloc[[31, 32], 3].sum()
df_pmd_18_19.iloc[30, 4] = df_pmd_18_19.iloc[[31,32], 4].sum()
df_pmd_18_19.iloc[30, 5] = df_pmd_18_19.iloc[[31, 32], 5].sum()
df_pmd_18_19.iloc[30, 6] = df_pmd_18_19.iloc[[31,32], 6].sum()

df_pmd_18_19

,County,Antlered_2017-18,Antlered_2018-19,Antlerless_2017-18,Antlerless_2018-19,Total_2017-18,Total_2018-19
0,Allegany,2078.0,2017.0,1283.0,1444.0,3361.0,3461.0
1,Anne Arundel,978.0,920.0,2001.0,1562.0,2979.0,2482.0
2,Baltimore,1754.0,1641.0,3805.0,2963.0,5559.0,4604.0
3,Calvert,558.0,614.0,1130.0,947.0,1688.0,1561.0
4,Caroline,757.0,829.0,1825.0,1728.0,2582.0,2557.0
5,whitetail,756.0,829.0,1824.0,1727.0,2580.0,2556.0
6,sika,1.0,0.0,1.0,1.0,2.0,1.0
7,Carroll,2116.0,2197.0,3780.0,3224.0,5896.0,5421.0
8,Cecil,1261.0,1237.0,2442.0,2269.0,3703.0,3506.0
9,Charles,1166.0,1031.0,2246.0,1328.0,3412.0,2359.0


In [50]:
#---------2018-2019 FINAL CLEANING: DROPPING ROWS---------#
#Filter county values for "whitetail" and "sika" and drop those rows
df_pmd_18_19=df_pmd_18_19[ df_pmd_18_19[ "County" ].str.contains( "whitetail|sika" )== False]
df_pmd_18_19 = df_pmd_18_19.reset_index(drop=True)
df_pmd_18_19.to_csv("df_pmd_18_19.csv")

In [51]:
#2017-2018 Processed Harvest Data Pt. 1

#PANDAS
df_pmd_17_18 = pd.read_html("https://news.maryland.gov/dnr/2018/02/15/maryland-hunters-harvest-86542-deer-in-2017-2018-season/", storage_options={"User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:52.0) Gecko/20100101 Firefox/52.0"})
df_pmd_17_18 = df_pmd_17_18[0]
#type(df_pmd_17_18)
df_pmd_18_19.to_csv("df_pmd_17_18.csv")


In [52]:
#2017-2018 Processed Harvest Data pt. 2 = COMBINING HEADERS

header_row_one_1718 = df_pmd_17_18.iloc[1]
header_row_two_1718 = df_pmd_17_18.iloc[2]

new_columns = []

for header_one, header_two in zip(header_row_one_1718, header_row_two_1718):
    if pd.isna(header_one):
    #https://pandas.pydata.org/docs/reference/api/pandas.isna.html 
        combined_header = header_two 
    else:
        combined_header = f"{header_one}_{header_two}"
    new_columns.append(combined_header)

#STORE NEW COMBINED HEADERS
df_pmd_17_18.columns = new_columns

#REPLACE OLD HEADERS WITH COMBINED
df_pmd_17_18 = df_pmd_17_18.iloc[3:]
df_pmd_17_18 = df_pmd_17_18.reset_index(drop=True)

df_pmd_17_18

#SAVE TO CSV (PROCESSED DATA FOLDER)
df_pmd_17_18.to_csv("df_pmd_17_18.csv")

In [53]:
#---------2017-2018 MORE CLEANING: REPLACING * , DROPPING LAST ROW, CHANGING DATATYPES, DROPPING % COLUMNS---------#

#FOR NULL COUNTY ROWS 

#1. Replace asterisks with NaN
df_pmd_17_18=df_pmd_17_18.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

#2. Drop last row, which just has "small sample size" text
df_pmd_17_18 = df_pmd_17_18.iloc[:-1]

#3. Change all datatypes (except those in county) to floats 
#found this on stack overflow to change datatypes of all but one column 

ignore = ['County']

#NEED TO SAVE THIS TO THE CURRENT DATAFRAME WITH DF=
df_pmd_17_18 = (df_pmd_17_18.set_index(ignore, append=True)
        .astype(float)
        .reset_index(ignore)
       )

# df_pmd_17_18.info()

#https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

#4.Drop percentage columns and then recalculate later 
df_pmd_17_18=df_pmd_17_18.drop(["Antlered_% Change", "Antlerless_% Change", "Total_% Change"], axis=1)

In [54]:
#---------2017-2018 MORE CLEANING: TEST TRIAL FOR COMBINING SUMS---------#
#5. Add whitetail and sika counts for each county manually, and then fill those sums into currently NaN county row 

#CAROLINE COUNTY
df_pmd_17_18.iloc[4, 1] = df_pmd_17_18.iloc[[5, 6], 1].sum()
df_pmd_17_18.iloc[4, 2] = df_pmd_17_18.iloc[[5,6], 2].sum()
df_pmd_17_18.iloc[4, 3] = df_pmd_17_18.iloc[[5, 6], 3].sum()
df_pmd_17_18.iloc[4, 4] = df_pmd_17_18.iloc[[5,6], 4].sum()
df_pmd_17_18.iloc[4, 5] = df_pmd_17_18.iloc[[5, 6], 5].sum()
df_pmd_17_18.iloc[4, 6] = df_pmd_17_18.iloc[[5,6], 6].sum()

df_pmd_17_18

,County,Antlered_’16-’17,Antlered_’17-’18,Antlerless_’16-’17,Antlerless_’17-’18,Total_’16-’17,Total_’17-’18
0,Allegany,1662.0,2078.0,1245.0,1283.0,2907.0,3361.0
1,Anne Arundel,900.0,978.0,1890.0,2001.0,2790.0,2979.0
2,Baltimore,1628.0,1754.0,3739.0,3805.0,5367.0,5559.0
3,Calvert,614.0,558.0,1253.0,1130.0,1867.0,1688.0
4,Caroline,918.0,757.0,2034.0,1825.0,2952.0,2582.0
5,whitetail,918.0,756.0,2033.0,1824.0,2951.0,2580.0
6,sika,0.0,1.0,1.0,1.0,1.0,2.0
7,Carroll,2002.0,2116.0,3661.0,3780.0,5663.0,5896.0
8,Cecil,1099.0,1261.0,2311.0,2442.0,3410.0,3703.0
9,Charles,1096.0,1166.0,2000.0,2246.0,3096.0,3412.0


In [55]:
#---------2017-2018 MORE CLEANING: COMBINING ALL SUMS FOR WHITETAIL AND SIKA FOR ALL APPLICABLE COUNTIES---------#

# #Dorchester
df_pmd_17_18.iloc[10, 1] = df_pmd_17_18.iloc[[11, 12], 1].sum()
df_pmd_17_18.iloc[10, 2] = df_pmd_17_18.iloc[[11,12], 2].sum()
df_pmd_17_18.iloc[10, 3] = df_pmd_17_18.iloc[[11, 12], 3].sum()
df_pmd_17_18.iloc[10, 4] = df_pmd_17_18.iloc[[11,12], 4].sum()
df_pmd_17_18.iloc[10, 5] = df_pmd_17_18.iloc[[11, 12], 5].sum()
df_pmd_17_18.iloc[10, 6] = df_pmd_17_18.iloc[[11,12], 6].sum()

# #Somerset
df_pmd_17_18.iloc[22, 1] = df_pmd_17_18.iloc[[23, 24], 1].sum()
df_pmd_17_18.iloc[22, 2] = df_pmd_17_18.iloc[[23,24], 2].sum()
df_pmd_17_18.iloc[22, 3] = df_pmd_17_18.iloc[[23, 24], 3].sum()
df_pmd_17_18.iloc[22, 4] = df_pmd_17_18.iloc[[23,24], 4].sum()
df_pmd_17_18.iloc[22, 5] = df_pmd_17_18.iloc[[23, 24], 5].sum()
df_pmd_17_18.iloc[22, 6] = df_pmd_17_18.iloc[[23,24], 6].sum()

# # #Wicomico
df_pmd_17_18.iloc[27, 1] = df_pmd_17_18.iloc[[28, 29], 1].sum()
df_pmd_17_18.iloc[27, 2] = df_pmd_17_18.iloc[[28,29], 2].sum()
df_pmd_17_18.iloc[27, 3] = df_pmd_17_18.iloc[[28, 29], 3].sum()
df_pmd_17_18.iloc[27, 4] = df_pmd_17_18.iloc[[28,29], 4].sum()
df_pmd_17_18.iloc[27, 5] = df_pmd_17_18.iloc[[28, 29], 5].sum()
df_pmd_17_18.iloc[27, 6] = df_pmd_17_18.iloc[[28,29], 6].sum()

# # #Worcester
df_pmd_17_18.iloc[30, 1] = df_pmd_17_18.iloc[[31, 32], 1].sum()
df_pmd_17_18.iloc[30, 2] = df_pmd_17_18.iloc[[31,32], 2].sum()
df_pmd_17_18.iloc[30, 3] = df_pmd_17_18.iloc[[31, 32], 3].sum()
df_pmd_17_18.iloc[30, 4] = df_pmd_17_18.iloc[[31,32], 4].sum()
df_pmd_17_18.iloc[30, 5] = df_pmd_17_18.iloc[[31, 32], 5].sum()
df_pmd_17_18.iloc[30, 6] = df_pmd_17_18.iloc[[31,32], 6].sum()

df_pmd_17_18

,County,Antlered_’16-’17,Antlered_’17-’18,Antlerless_’16-’17,Antlerless_’17-’18,Total_’16-’17,Total_’17-’18
0,Allegany,1662.0,2078.0,1245.0,1283.0,2907.0,3361.0
1,Anne Arundel,900.0,978.0,1890.0,2001.0,2790.0,2979.0
2,Baltimore,1628.0,1754.0,3739.0,3805.0,5367.0,5559.0
3,Calvert,614.0,558.0,1253.0,1130.0,1867.0,1688.0
4,Caroline,918.0,757.0,2034.0,1825.0,2952.0,2582.0
5,whitetail,918.0,756.0,2033.0,1824.0,2951.0,2580.0
6,sika,0.0,1.0,1.0,1.0,1.0,2.0
7,Carroll,2002.0,2116.0,3661.0,3780.0,5663.0,5896.0
8,Cecil,1099.0,1261.0,2311.0,2442.0,3410.0,3703.0
9,Charles,1096.0,1166.0,2000.0,2246.0,3096.0,3412.0


In [56]:
#---------2017-2018 FINAL CLEANING: DROPPING ROWS---------#
#Filter county values for "whitetail" and "sika" and drop those rows
df_pmd_17_18=df_pmd_17_18[ df_pmd_17_18[ "County" ].str.contains( "whitetail|sika" )== False]
df_pmd_17_18 = df_pmd_17_18.reset_index(drop=True)
df_pmd_17_18.to_csv("df_pmd_17_18.csv")

In [57]:
#2016-2017 Processed Harvest Data Pt. 1

#PANDAS
df_pmd_16_17 = pd.read_html("https://news.maryland.gov/dnr/2017/02/16/85193-deer-harvested-during-2016-2017-hunting-season/", storage_options={"User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:52.0) Gecko/20100101 Firefox/52.0"})
df_pmd_16_17 = df_pmd_16_17[0]
#type(df_pmd_16_17)
df_pmd_16_17.to_csv("df_pmd_16_17.csv")

In [58]:
#2016-2017 Processed Harvest Data pt. 2 = COMBINING HEADERS

header_row_one_1617 = df_pmd_16_17.iloc[1]
header_row_two_1617 = df_pmd_16_17.iloc[2]

new_columns = []

for header_one, header_two in zip(header_row_one_1617, header_row_two_1617):
    if pd.isna(header_one):
    #https://pandas.pydata.org/docs/reference/api/pandas.isna.html 
        combined_header = header_two 
    else:
        combined_header = f"{header_one}_{header_two}"
    new_columns.append(combined_header)

#STORE NEW COMBINED HEADERS
df_pmd_16_17.columns = new_columns

#REPLACE OLD HEADERS WITH COMBINED
df_pmd_16_17 = df_pmd_16_17.iloc[4:]
#this year's report formatted differently. Extra header row and a bunch of extra null columns 

df_pmd_16_17 = df_pmd_16_17.reset_index(drop=True)

df_pmd_16_17

,County,Antlered_nan,Antlered_2016-17,Antlered_% Change,NaN,Antlerless_nan,Antlerless_2016-17,Antlerless_% Change,NaN,Total_nan,Total_2016-17,Total_% Change,NaN
0,Allegany,NaN,1662,-16.2,NaN,NaN,1245,-10.0,NaN,NaN,2907,-13.7,NaN
1,Anne Arundel,NaN,900,2.0,NaN,NaN,1890,4.4,NaN,NaN,2790,3.6,NaN
2,Baltimore,NaN,1628,11.3,NaN,NaN,3739,6.6,NaN,NaN,5367,8.0,NaN
3,Calvert,NaN,614,4.8,NaN,NaN,1253,16.1,NaN,NaN,1867,12.1,NaN
4,Caroline,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,whitetail,NaN,918,13.5,NaN,NaN,2033,2.2,NaN,NaN,2951,5.5,NaN
6,sika,NaN,0,*,NaN,NaN,1,*,NaN,NaN,1,*,NaN
7,Carroll,NaN,2002,10.6,NaN,NaN,3661,6.8,NaN,NaN,5663,8.1,NaN
8,Cecil,NaN,1099,6.4,NaN,NaN,2311,10.3,NaN,NaN,3410,9.0,NaN
9,Charles,NaN,1096,2.2,NaN,NaN,2000,-2.5,NaN,NaN,3096,-0.9,NaN


In [59]:
#---------2016-2017 MORE CLEANING: REPLACING * , DROPPING LAST ROW, CHANGING DATATYPES, DROPPING % COLUMNS---------#

#FOR NULL COUNTY ROWS 

#1. Replace asterisks with NaN
df_pmd_16_17=df_pmd_16_17.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

#2. Drop last row, which just has "small sample size" text
df_pmd_16_17 = df_pmd_16_17.iloc[:-1]

#3. Change all datatypes (except those in county) to floats 
#found this on stack overflow to change datatypes of all but one column 

ignore = ['County']

#NEED TO SAVE THIS TO THE CURRENT DATAFRAME WITH DF=
df_pmd_16_17 = (df_pmd_16_17.set_index(ignore, append=True)
        .astype(float)
        .reset_index(ignore)
       )

df_pmd_16_17.columns

#https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

#4.Drop percentage columns and then recalculate later 
#This table had additional columns to drop. Here are all the ones that were dropped, or rather not kept: ['Antlered_nan',
#'Antlered_% Change', nan, 'Antlerless_nan', 'Antlerless_% Change', nan, 'Total_nan', 'Total_% Change', nan]

df_pmd_16_17=df_pmd_16_17.loc[:, ['County', 'Antlered_2016-17', 'Antlerless_2016-17', 'Total_2016-17']]
#this table doesn't inlude values from previous years like the later tables do 
df_pmd_16_17

,County,Antlered_2016-17,Antlerless_2016-17,Total_2016-17
0,Allegany,1662.0,1245.0,2907.0
1,Anne Arundel,900.0,1890.0,2790.0
2,Baltimore,1628.0,3739.0,5367.0
3,Calvert,614.0,1253.0,1867.0
4,Caroline,NaN,NaN,NaN
5,whitetail,918.0,2033.0,2951.0
6,sika,0.0,1.0,1.0
7,Carroll,2002.0,3661.0,5663.0
8,Cecil,1099.0,2311.0,3410.0
9,Charles,1096.0,2000.0,3096.0


In [60]:
#---------2016-2017 MORE CLEANING: TEST TRIAL FOR COMBINING SUMS---------#
#5. Add whitetail and sika counts for each county manually, and then fill those sums into currently NaN county row 

#CAROLINE COUNTY
df_pmd_16_17.iloc[4, 1] = df_pmd_16_17.iloc[[5, 6], 1].sum()
df_pmd_16_17.iloc[4, 2] = df_pmd_16_17.iloc[[5,6], 2].sum()
df_pmd_16_17.iloc[4, 3] = df_pmd_16_17.iloc[[5, 6], 3].sum()

df_pmd_16_17

,County,Antlered_2016-17,Antlerless_2016-17,Total_2016-17
0,Allegany,1662.0,1245.0,2907.0
1,Anne Arundel,900.0,1890.0,2790.0
2,Baltimore,1628.0,3739.0,5367.0
3,Calvert,614.0,1253.0,1867.0
4,Caroline,918.0,2034.0,2952.0
5,whitetail,918.0,2033.0,2951.0
6,sika,0.0,1.0,1.0
7,Carroll,2002.0,3661.0,5663.0
8,Cecil,1099.0,2311.0,3410.0
9,Charles,1096.0,2000.0,3096.0


In [61]:
#---------2016-2017 MORE CLEANING: COMBINING ALL SUMS FOR WHITETAIL AND SIKA FOR ALL APPLICABLE COUNTIES---------#

# #Dorchester
df_pmd_16_17.iloc[10, 1] = df_pmd_16_17.iloc[[11, 12], 1].sum()
df_pmd_16_17.iloc[10, 2] = df_pmd_16_17.iloc[[11,12], 2].sum()
df_pmd_16_17.iloc[10, 3] = df_pmd_16_17.iloc[[11, 12], 3].sum()

# #Somerset
df_pmd_16_17.iloc[22, 1] = df_pmd_16_17.iloc[[23, 24], 1].sum()
df_pmd_16_17.iloc[22, 2] = df_pmd_16_17.iloc[[23,24], 2].sum()
df_pmd_16_17.iloc[22, 3] = df_pmd_16_17.iloc[[23, 24], 3].sum()

# # #Wicomico
df_pmd_16_17.iloc[27, 1] = df_pmd_16_17.iloc[[28, 29], 1].sum()
df_pmd_16_17.iloc[27, 2] = df_pmd_16_17.iloc[[28,29], 2].sum()
df_pmd_16_17.iloc[27, 3] = df_pmd_16_17.iloc[[28, 29], 3].sum()

# # #Worcester
df_pmd_16_17.iloc[30, 1] = df_pmd_16_17.iloc[[31, 32], 1].sum()
df_pmd_16_17.iloc[30, 2] = df_pmd_16_17.iloc[[31,32], 2].sum()
df_pmd_16_17.iloc[30, 3] = df_pmd_16_17.iloc[[31, 32], 3].sum()

df_pmd_16_17

,County,Antlered_2016-17,Antlerless_2016-17,Total_2016-17
0,Allegany,1662.0,1245.0,2907.0
1,Anne Arundel,900.0,1890.0,2790.0
2,Baltimore,1628.0,3739.0,5367.0
3,Calvert,614.0,1253.0,1867.0
4,Caroline,918.0,2034.0,2952.0
5,whitetail,918.0,2033.0,2951.0
6,sika,0.0,1.0,1.0
7,Carroll,2002.0,3661.0,5663.0
8,Cecil,1099.0,2311.0,3410.0
9,Charles,1096.0,2000.0,3096.0


In [62]:
#---------2016-2017 FINAL CLEANING: DROPPING ROWS---------#
#Filter county values for "whitetail" and "sika" and drop those rows
df_pmd_16_17=df_pmd_16_17[ df_pmd_16_17[ "County" ].str.contains( "whitetail|sika" )== False]
df_pmd_16_17 = df_pmd_16_17.reset_index(drop=True)
df_pmd_16_17.to_csv("df_pmd_16_17.csv")

In [63]:
#2015-2016 Processed Harvest Data Pt. 1

#PANDAS
df_pmd_15_16 = pd.read_html("https://news.maryland.gov/dnr/2016/02/09/84022-deer-harvested-in-maryland-during-2015-2016-hunting-season/", storage_options={"User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:52.0) Gecko/20100101 Firefox/52.0"})
df_pmd_15_16 = df_pmd_15_16[0]
#type(df_pmd_15_16)
df_pmd_15_16.to_csv("df_pmd_15_16.csv")

In [64]:
#2015-2016 Processed Harvest Data pt. 2 = COMBINING HEADERS

header_row_one_1516 = df_pmd_15_16.iloc[1]
header_row_two_1516 = df_pmd_15_16.iloc[2]

new_columns = []

for header_one, header_two in zip(header_row_one_1516, header_row_two_1516):
    if pd.isna(header_one):
    #https://pandas.pydata.org/docs/reference/api/pandas.isna.html 
        combined_header = header_two 
    else:
        combined_header = f"{header_one}_{header_two}"
    new_columns.append(combined_header)

#STORE NEW COMBINED HEADERS
df_pmd_15_16.columns = new_columns

#REPLACE OLD HEADERS WITH COMBINED
df_pmd_15_16 = df_pmd_15_16.iloc[4:]
df_pmd_15_16 = df_pmd_15_16.reset_index(drop=True)

df_pmd_15_16

,County,Antlered_2014-15,Antlered_2015-16,Antlered_% Change,NaN,Antlerless_2014-15,Antlerless_2015-16,Antlerless_% Change,NaN,Total_2014-15,Total_2015-16,Total_% Change,NaN
0,Allegany,1731,1984,14.6,NaN,1320,1384,4.8,NaN,3051,3368,10.4,NaN
1,Anne Arundel,817,882,8.0,NaN,2075,1810,-12.8,NaN,2892,2692,-6.9,NaN
2,Baltimore,1502,1463,-2.6,NaN,3911,3507,-10.3,NaN,5413,4970,-8.2,NaN
3,Calvert,470,586,24.7,NaN,1101,1079,-2.0,NaN,1571,1665,6.0,NaN
4,Caroline,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,whitetail,734,809,10.2,NaN,1704,1989,16.7,NaN,2438,2798,14.8,NaN
6,sika,1,0,*,NaN,1,2,*,NaN,2,2,*,NaN
7,Carroll,1634,1810,10.8,NaN,3830,3428,-10.5,NaN,5464,5238,-4.1,NaN
8,Cecil,1005,1033,2.8,NaN,2455,2095,-14.7,NaN,3460,3128,-9.6,NaN
9,Charles,1132,1072,-5.3,NaN,2392,2051,-14.3,NaN,3524,3123,-11.4,NaN


In [65]:
#---------2015-2016 MORE CLEANING: REPLACING * , DROPPING LAST ROW, CHANGING DATATYPES, DROPPING % COLUMNS---------#

#FOR NULL COUNTY ROWS 

#1. Replace asterisks with NaN
df_pmd_15_16=df_pmd_15_16.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

#2. Drop last row, which just has "small sample size" text
df_pmd_15_16 = df_pmd_15_16.iloc[:-1]

#3. Change all datatypes (except those in county) to floats 
#found this on stack overflow to change datatypes of all but one column 

ignore = ['County']

#NEED TO SAVE THIS TO THE CURRENT DATAFRAME WITH DF=
df_pmd_15_16 = (df_pmd_15_16.set_index(ignore, append=True)
        .astype(float)
        .reset_index(ignore)
       )

# #https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

#df_pmd_15_16.columns

    #Index(['County', 'Antlered_2014-15', 'Antlered_2015-16', 'Antlered_% Change',
          #  nan, 'Antlerless_2014-15', 'Antlerless_2015-16', 'Antlerless_% Change',
          #  nan, 'Total_2014-15', 'Total_2015-16', 'Total_% Change', nan]

#4.Drop percentage columns and then recalculate later 
#This table had additional columns to drop. Here are all the ones that were dropped, or rather not kept: ['Antlered_% Change',
          #  nan, 'Antlerless_% Change', nan, Total_% Change', nan]

df_pmd_15_16=df_pmd_15_16.loc[:, ['County', 'Antlered_2014-15', 'Antlered_2015-16', 'Antlerless_2014-15', 'Antlerless_2015-16', 
                                  'Total_2014-15', 'Total_2015-16']] 
df_pmd_15_16

,County,Antlered_2014-15,Antlered_2015-16,Antlerless_2014-15,Antlerless_2015-16,Total_2014-15,Total_2015-16
0,Allegany,1731.0,1984.0,1320.0,1384.0,3051.0,3368.0
1,Anne Arundel,817.0,882.0,2075.0,1810.0,2892.0,2692.0
2,Baltimore,1502.0,1463.0,3911.0,3507.0,5413.0,4970.0
3,Calvert,470.0,586.0,1101.0,1079.0,1571.0,1665.0
4,Caroline,NaN,NaN,NaN,NaN,NaN,NaN
5,whitetail,734.0,809.0,1704.0,1989.0,2438.0,2798.0
6,sika,1.0,0.0,1.0,2.0,2.0,2.0
7,Carroll,1634.0,1810.0,3830.0,3428.0,5464.0,5238.0
8,Cecil,1005.0,1033.0,2455.0,2095.0,3460.0,3128.0
9,Charles,1132.0,1072.0,2392.0,2051.0,3524.0,3123.0


In [66]:
#---------2015-2016 MORE CLEANING: TEST TRIAL FOR COMBINING SUMS---------#
#5. Add whitetail and sika counts for each county manually, and then fill those sums into currently NaN county row 

#CAROLINE COUNTY
df_pmd_15_16.iloc[4, 1] = df_pmd_15_16.iloc[[5, 6], 1].sum()
df_pmd_15_16.iloc[4, 2] = df_pmd_15_16.iloc[[5,6], 2].sum()
df_pmd_15_16.iloc[4, 3] = df_pmd_15_16.iloc[[5, 6], 3].sum()
df_pmd_15_16.iloc[4, 4] = df_pmd_15_16.iloc[[5,6], 4].sum()
df_pmd_15_16.iloc[4, 5] = df_pmd_15_16.iloc[[5, 6], 5].sum()
df_pmd_15_16.iloc[4, 6] = df_pmd_15_16.iloc[[5,6], 6].sum()

df_pmd_15_16

,County,Antlered_2014-15,Antlered_2015-16,Antlerless_2014-15,Antlerless_2015-16,Total_2014-15,Total_2015-16
0,Allegany,1731.0,1984.0,1320.0,1384.0,3051.0,3368.0
1,Anne Arundel,817.0,882.0,2075.0,1810.0,2892.0,2692.0
2,Baltimore,1502.0,1463.0,3911.0,3507.0,5413.0,4970.0
3,Calvert,470.0,586.0,1101.0,1079.0,1571.0,1665.0
4,Caroline,735.0,809.0,1705.0,1991.0,2440.0,2800.0
5,whitetail,734.0,809.0,1704.0,1989.0,2438.0,2798.0
6,sika,1.0,0.0,1.0,2.0,2.0,2.0
7,Carroll,1634.0,1810.0,3830.0,3428.0,5464.0,5238.0
8,Cecil,1005.0,1033.0,2455.0,2095.0,3460.0,3128.0
9,Charles,1132.0,1072.0,2392.0,2051.0,3524.0,3123.0


In [67]:
#---------2015-2016 MORE CLEANING: COMBINING ALL SUMS FOR WHITETAIL AND SIKA FOR ALL APPLICABLE COUNTIES---------#

# #Dorchester
df_pmd_15_16.iloc[10, 1] = df_pmd_15_16.iloc[[11, 12], 1].sum()
df_pmd_15_16.iloc[10, 2] = df_pmd_15_16.iloc[[11,12], 2].sum()
df_pmd_15_16.iloc[10, 3] = df_pmd_15_16.iloc[[11, 12], 3].sum()
df_pmd_15_16.iloc[10, 4] = df_pmd_15_16.iloc[[11,12], 4].sum()
df_pmd_15_16.iloc[10, 5] = df_pmd_15_16.iloc[[11, 12], 5].sum()
df_pmd_15_16.iloc[10, 6] = df_pmd_15_16.iloc[[11,12], 6].sum()

# #Somerset
df_pmd_15_16.iloc[22, 1] = df_pmd_15_16.iloc[[23, 24], 1].sum()
df_pmd_15_16.iloc[22, 2] = df_pmd_15_16.iloc[[23,24], 2].sum()
df_pmd_15_16.iloc[22, 3] = df_pmd_15_16.iloc[[23, 24], 3].sum()
df_pmd_15_16.iloc[22, 4] = df_pmd_15_16.iloc[[23,24], 4].sum()
df_pmd_15_16.iloc[22, 5] = df_pmd_15_16.iloc[[23, 24], 5].sum()
df_pmd_15_16.iloc[22, 6] = df_pmd_15_16.iloc[[23,24], 6].sum()

# # #Wicomico
df_pmd_15_16.iloc[27, 1] = df_pmd_15_16.iloc[[28, 29], 1].sum()
df_pmd_15_16.iloc[27, 2] = df_pmd_15_16.iloc[[28,29], 2].sum()
df_pmd_15_16.iloc[27, 3] = df_pmd_15_16.iloc[[28, 29], 3].sum()
df_pmd_15_16.iloc[27, 4] = df_pmd_15_16.iloc[[28,29], 4].sum()
df_pmd_15_16.iloc[27, 5] = df_pmd_15_16.iloc[[28, 29], 5].sum()
df_pmd_15_16.iloc[27, 6] = df_pmd_15_16.iloc[[28,29], 6].sum()

# # #Worcester
df_pmd_15_16.iloc[30, 1] = df_pmd_15_16.iloc[[31, 32], 1].sum()
df_pmd_15_16.iloc[30, 2] = df_pmd_15_16.iloc[[31,32], 2].sum()
df_pmd_15_16.iloc[30, 3] = df_pmd_15_16.iloc[[31, 32], 3].sum()
df_pmd_15_16.iloc[30, 4] = df_pmd_15_16.iloc[[31,32], 4].sum()
df_pmd_15_16.iloc[30, 5] = df_pmd_15_16.iloc[[31, 32], 5].sum()
df_pmd_15_16.iloc[30, 6] = df_pmd_15_16.iloc[[31,32], 6].sum()

df_pmd_15_16

,County,Antlered_2014-15,Antlered_2015-16,Antlerless_2014-15,Antlerless_2015-16,Total_2014-15,Total_2015-16
0,Allegany,1731.0,1984.0,1320.0,1384.0,3051.0,3368.0
1,Anne Arundel,817.0,882.0,2075.0,1810.0,2892.0,2692.0
2,Baltimore,1502.0,1463.0,3911.0,3507.0,5413.0,4970.0
3,Calvert,470.0,586.0,1101.0,1079.0,1571.0,1665.0
4,Caroline,735.0,809.0,1705.0,1991.0,2440.0,2800.0
5,whitetail,734.0,809.0,1704.0,1989.0,2438.0,2798.0
6,sika,1.0,0.0,1.0,2.0,2.0,2.0
7,Carroll,1634.0,1810.0,3830.0,3428.0,5464.0,5238.0
8,Cecil,1005.0,1033.0,2455.0,2095.0,3460.0,3128.0
9,Charles,1132.0,1072.0,2392.0,2051.0,3524.0,3123.0


In [68]:
#---------2015-2016 FINAL CLEANING: DROPPING ROWS---------#
#Filter county values for "whitetail" and "sika" and drop those rows and reset index
df_pmd_15_16=df_pmd_15_16[ df_pmd_15_16[ "County" ].str.contains( "whitetail|sika" )== False]
df_pmd_15_16= df_pmd_15_16.reset_index(drop=True)
df_pmd_15_16.to_csv("df_pmd_15_16.csv")

In [164]:
#---------COMBINE THROUGH CONCAT, STANDARDIZE LABELS, DOUBLE-CHECK FOR CONSISTENT SPELLING OF COUNTY NAMES---------#

df_combined_harvest = pd.concat([df_pmd_15_16, df_pmd_16_17, df_pmd_17_18, df_pmd_18_19, df_pmd_19_20,
           df_pmd_20_21, df_pmd_21_22, df_pmd_22_23, df_pmd_23_24, df_pmd_24_25, df_pmd_25_26])

df_combined_harvest.columns

#COLUMN LABELS:
    #County
    #Antlered_2014-15
    #Antlerless_2014-15
    #Total_2014-15

#RENAME COLUMNS AS NEEDED TO FOLLOW CONVENTION ABOVE:
#https://www.geeksforgeeks.org/python/how-to-rename-multiple-column-headers-in-a-pandas-dataframe/
# create a dictionary
# key = old name
# value = new name
dict = {'Antlered_’16-’17': 'Antlered_2016-17',
        'Antlered_’17-’18': 'Antlered_2017-18',
        'Antlerless_’16-’17': 'Antlerless_2016-17',
        'Antlerless_’17-’18': 'Antlerless_2017-18',
        'Total_’16-’17': 'Total_2016-17',
        'Total_’17-’18': 'Total_2017-18',
       }
df_combined_harvest.rename(columns=dict,
          inplace=True)
df_combined_harvest.to_csv("df_combined_harvest.csv", index=False)


In [166]:
#DROP "TOTAL" ROWS
df_combined_harvest=df_combined_harvest[ df_combined_harvest[ "County" ].str.contains( "Total" )== False]
df_combined_harvest = df_combined_harvest.reset_index(drop=True)
df_combined_harvest.to_csv("df_combined_harvest.csv", index=False)

In [70]:
# Create population DF based on list of estimated deer population from DNR, Carson Coriell

import csv

md_deer_pop_est_15_25_fromCarson = [236921, 242761, 251609, 223745, 239444, 246919, 264760, 270118, 261832, 268863]

# Column headers
fields = ['Hunting_Season', 'Est_Deer_Population']

# Data rows
rows = [ ['2015-16', '236921'],
         ['2016-17', '242761'],
         ['2017-18', '251609'],
         ['2018-19', '223745'],
         ['2019-20', '239444'],
         ['2020-21', '246919'],
         ['2021-22', '264760'],
         ['2022-23', '270118'],
         ['2023-24', '261832'],
         ['2024-25', '268863'],
         ['2025-26', '268863']
       ]

# Writing to a CSV file
with open('md_deer_pop_est_15_25_fromCarson', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(fields)     # Write header
    writer.writerows(rows)      # Write data rows
    
md_deer_pop_est_15_25_fromCarson = pd.read_csv("md_deer_pop_est_15_25_fromCarson")

# 10-yr White-tail Pop. Estimates
# Year	Population Estimate     
# 2015	236,921
# 2016	242,761
# 2017	251,609
# 2018	223,745
# 2019	239,444
# 2020	246,919
# 2021	264,760
# 2022	270,118
# 2023	261,832
# 2024	268,863

md_deer_pop_est_15_25_fromCarson.to_csv("md_deer_pop_est_15_25_fromCarson.csv")

In [72]:
# #--------Steps to clean button buck and doe harvest numbers--------#

# #1. FORMAT OF COUNTY NAMES:
#     #strip \n
#     #Anne Arundel
#     #Prince George's 
#     #Queen Anne's

# #2. Replace asterisks with NaN
# # df_pmd_22_23=df_pmd_22_23.replace(r'^\*$', np.nan, regex=True)
# ##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

# #3. Drop last row, which just has "small sample size" text
# #df_pmd_22_23 = df_pmd_22_23.iloc[:-1]

# #4. Change all datatypes (except those in county) to floats 
# #found this on stack overflow to change datatypes of all but one column 

# ignore = ['County']

# #NEED TO SAVE THIS TO THE CURRENT DATAFRAME WITH DF=
# # df_pmd_22_23 = (df_pmd_22_23.set_index(ignore, append=True)
# #         .astype(float)
# #         .reset_index(ignore)
# #        )

# # df_pmd_22_23.info()

# #https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

# #5. Rename columns
#     #County
#     #Buttonbuck_2015-16
#     #Female_or_Antlerless_2015-16
#     #Total_2015-16

# #6. Drop percent in buttonbuck
# df_pmd_22_23=df_pmd_22_23.drop(["Antlered_% Change", "Antlerless_% Change", "Total_% Change"], axis=1)

In [73]:
#2015-2016 Button buck and doe harvest 

from natural_pdf import PDF

# Open a PDF
pdf = PDF('md_annual_deer_report15-16.pdf')
#had to download PDF bc of 403 error 
page = pdf.pages[11]
page.show(width=700)

table = page.extract_table(method="pdfplumber")
list(table)

import pandas as pd

rows = list(table)

df_pbutton_15_16 = pd.DataFrame(rows)

df_pbutton_15_16.to_csv("df_pbutton_15_16.csv", index=False)

In [74]:
#--------2015-2016 Button Buck and Doe Harvest Cleaning Pt. 1--------#

#1. RENAME COLUMNS BY INDEX
#https://www.geeksforgeeks.org/python/rename-column-by-index-in-pandas/

df_pbutton_15_16.rename(columns={
    df_pbutton_15_16.columns[0]: "County",
    df_pbutton_15_16.columns[1]: "Buttonbuck_2015-16",
    df_pbutton_15_16.columns[2]: "Female_or_Antlerless_2015-16",
    df_pbutton_15_16.columns[3]: "Total_2015-16",
    df_pbutton_15_16.columns[4]: "Percent_Buttonbuck"
    
}, inplace=True)
 
#2. FORMAT OF COUNTY NAMES:
    #strip \n
    #Anne Arundel
    #Prince George's 
    #Queen Anne's
df_pbutton_15_16['County']=df_pbutton_15_16['County'].str.replace("\n", " ")
#https://stackoverflow.com/questions/65666852/why-cant-i-replace-a-newline-in-my-pandas-dataframe

# #https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

# 3. Drop percent in buttonbuck
df_pbutton_15_16.drop(df_pbutton_15_16.columns[4], axis=1, inplace = True)

In [75]:
df_pbutton_15_16

,County,Buttonbuck_2015-16,Female_or_Antlerless_2015-16,Total_2015-16
0,County,Buttonbuck,Female or\nAntlerless,Total
1,Allegany,192,1192,1384
2,Anne Arundel,305,1505,1810
3,Baltimore,436,3071,3507
4,Calvert,173,906,1079
5,Caroline,,,
6,Whitetail,329,1660,1989
7,Sika,0,2,2
8,Carroll,429,2999,3428
9,Cecil,325,1770,2095


In [76]:
#--------2015-2016 Button Buck and Doe Harvest Cleaning Pt. 2--------#
import numpy as np 
# 4. Replace asterisks and spaces with NaN
df_pbutton_15_16=df_pbutton_15_16.replace(r'^\*$', np.nan, regex=True)
#https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

In [77]:
#5. Drop last row, which just has "small sample size" text
df_pbutton_15_16=df_pbutton_15_16.drop(df_pbutton_15_16.index[35])

In [78]:
df_pbutton_15_16

,County,Buttonbuck_2015-16,Female_or_Antlerless_2015-16,Total_2015-16
0,County,Buttonbuck,Female or\nAntlerless,Total
1,Allegany,192,1192,1384
2,Anne Arundel,305,1505,1810
3,Baltimore,436,3071,3507
4,Calvert,173,906,1079
5,Caroline,,,
6,Whitetail,329,1660,1989
7,Sika,0,2,2
8,Carroll,429,2999,3428
9,Cecil,325,1770,2095


In [79]:
#6. Drop first row
df_pbutton_15_16.drop(labels=0, inplace=True)

#7. Strip commas
df_pbutton_15_16.replace(',','', regex=True, inplace=True)

,County,Buttonbuck_2015-16,Female_or_Antlerless_2015-16,Total_2015-16
1,Allegany,192,1192,1384
2,Anne Arundel,305,1505,1810
3,Baltimore,436,3071,3507
4,Calvert,173,906,1079
5,Caroline,,,
6,Whitetail,329,1660,1989
7,Sika,0,2,2
8,Carroll,429,2999,3428
9,Cecil,325,1770,2095
10,Charles,302,1749,2051


In [80]:
df_pbutton_15_16.iloc[4]

County                          Caroline
Buttonbuck_2015-16                      
Female_or_Antlerless_2015-16            
Total_2015-16                           
Name: 5, dtype: str

In [81]:
#--------2015-2016 Button Buck and Doe Harvest Cleaning Pt. 3--------#

#8. Change datatype from string to float for numeric avlues 
df_pbutton_15_16['Buttonbuck_2015-16'] = pd.to_numeric( df_pbutton_15_16['Buttonbuck_2015-16'], errors="coerce")
df_pbutton_15_16['Female_or_Antlerless_2015-16'] = pd.to_numeric( df_pbutton_15_16['Female_or_Antlerless_2015-16'], errors="coerce")
df_pbutton_15_16['Total_2015-16'] = pd.to_numeric( df_pbutton_15_16['Total_2015-16'], errors="coerce")

#9. Reset index after dropping rows. 
df_pbutton_15_16 = df_pbutton_15_16.reset_index(drop=True) 

In [82]:
#--------2015-2016 Button Buck and Doe Harvest Cleaning Pt. 3--------#

#10. Sum whitetail and sika, then move those sums to county row

#CAROLINE COUNTY
df_pbutton_15_16 = df_pbutton_15_16.reset_index(drop=True) 
df_pbutton_15_16.iloc[4, 1] = df_pbutton_15_16.iloc[[5, 6], 1].sum()
df_pbutton_15_16.iloc[4, 2] = df_pbutton_15_16.iloc[[5,6], 2].sum()
df_pbutton_15_16.iloc[4, 3] = df_pbutton_15_16.iloc[[5, 6], 3].sum()

# #Dorchester
df_pbutton_15_16.iloc[10, 1] = df_pbutton_15_16.iloc[[11, 12], 1].sum()
df_pbutton_15_16.iloc[10, 2] = df_pbutton_15_16.iloc[[11,12], 2].sum()
df_pbutton_15_16.iloc[10, 3] = df_pbutton_15_16.iloc[[11, 12], 3].sum()

#Somerset
df_pbutton_15_16.iloc[22, 1] = df_pbutton_15_16.iloc[[23, 24], 1].sum()
df_pbutton_15_16.iloc[22, 2] = df_pbutton_15_16.iloc[[23,24], 2].sum()
df_pbutton_15_16.iloc[22, 3] = df_pbutton_15_16.iloc[[23, 24], 3].sum()

#Wicomico
df_pbutton_15_16.iloc[27, 1] = df_pbutton_15_16.iloc[[28, 29], 1].sum()
df_pbutton_15_16.iloc[27, 2] = df_pbutton_15_16.iloc[[28,29], 2].sum()
df_pbutton_15_16.iloc[27, 3] = df_pbutton_15_16.iloc[[28, 29], 3].sum()

# #Worcester
df_pbutton_15_16.iloc[30, 1] = df_pbutton_15_16.iloc[[31, 32], 1].sum()
df_pbutton_15_16.iloc[30, 2] = df_pbutton_15_16.iloc[[31,32], 2].sum()
df_pbutton_15_16.iloc[30, 3] = df_pbutton_15_16.iloc[[31, 32], 3].sum()

In [83]:
#---------2015-2016 Button Buck and Doe Harvest FINAL CLEANING: DROPPING ROWS---------#
#11. Filter county values for "whitetail" and "sika" and drop those rows
df_pbutton_15_16=df_pbutton_15_16[ df_pbutton_15_16[ "County" ].str.contains( "Whitetail|Sika" )== False]
df_pbutton_15_16= df_pbutton_15_16.reset_index(drop=True)
df_pbutton_15_16.to_csv("df_pbutton_15_16.csv")

In [84]:
df_pbutton_15_16

,County,Buttonbuck_2015-16,Female_or_Antlerless_2015-16,Total_2015-16
0,Allegany,192.0,1192.0,1384.0
1,Anne Arundel,305.0,1505.0,1810.0
2,Baltimore,436.0,3071.0,3507.0
3,Calvert,173.0,906.0,1079.0
4,Caroline,329.0,1662.0,1991.0
5,Carroll,429.0,2999.0,3428.0
6,Cecil,325.0,1770.0,2095.0
7,Charles,302.0,1749.0,2051.0
8,Dorchester,436.0,2834.0,3270.0
9,Frederick,514.0,4082.0,4596.0


In [85]:
#2016-2017 Button buck and doe harvest 

# Open a PDF
pdf = PDF('md_annual_deer_report16-17.pdf')
page = pdf.pages[11]
page.show(width=700)

table = page.extract_table(method="pdfplumber")
list(table)

rows = list(table)

df_pbutton_16_17 = pd.DataFrame(rows)

df_pbutton_16_17.to_csv("df_pbutton_16_17.csv", index=False)

In [86]:
df_pbutton_16_17

,0,1,2,3,4
0,COUNTY,Buttonbuck,Female or\nAntlerless,Total,Percent\nButtonbuck
1,Allegany,174,1071,1245,14.0
2,Anne\nArundel,310,1580,1890,16.4
3,Baltimore,425,3314,3739,11.4
4,Calvert,193,1060,1253,15.4
5,Caroline,,,,
6,Whitetail,338,1695,2033,16.6
7,Sika,0,1,1,*
8,Carroll,505,3156,3661,13.8
9,Cecil,342,1969,2311,14.8


In [87]:
#--------2016-2017 Button Buck and Doe Harvest Cleaning Pt. 1--------#

#1. RENAME COLUMNS BY INDEX
#https://www.geeksforgeeks.org/python/rename-column-by-index-in-pandas/

df_pbutton_16_17.rename(columns={
    df_pbutton_16_17.columns[0]: "County",
    df_pbutton_16_17.columns[1]: "Buttonbuck_2016-17",
    df_pbutton_16_17.columns[2]: "Female_or_Antlerless_2016-17",
    df_pbutton_16_17.columns[3]: "Total_2016-17",
    df_pbutton_16_17.columns[4]: "Percent_Buttonbuck"
    
}, inplace=True)
 
#2. FORMAT OF COUNTY NAMES:
    #strip \n
    #Anne Arundel
    #Prince George's 
    #Queen Anne's
df_pbutton_16_17['County']=df_pbutton_16_17['County'].str.replace("\n", " ")
#https://stackoverflow.com/questions/65666852/why-cant-i-replace-a-newline-in-my-pandas-dataframe
#https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

#3. Drop percent in buttonbuck
df_pbutton_16_17 = df_pbutton_16_17.drop(df_pbutton_16_17.columns[4], axis=1)

In [88]:
df_pbutton_16_17

,County,Buttonbuck_2016-17,Female_or_Antlerless_2016-17,Total_2016-17
0,COUNTY,Buttonbuck,Female or\nAntlerless,Total
1,Allegany,174,1071,1245
2,Anne Arundel,310,1580,1890
3,Baltimore,425,3314,3739
4,Calvert,193,1060,1253
5,Caroline,,,
6,Whitetail,338,1695,2033
7,Sika,0,1,1
8,Carroll,505,3156,3661
9,Cecil,342,1969,2311


In [89]:
#--------2016-2017 Button Buck and Doe Harvest Cleaning Pt. 2--------#

#4. Replace asterisks and spaces with NaN
df_pbutton_16_17=df_pbutton_16_17.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

#5. Drop last row, which just has "small sample size" text
df_pbutton_16_17 = df_pbutton_16_17.iloc[:-1]

#6. Drop first row
df_pbutton_16_17.drop(labels=0, inplace=True)

#7. Strip commas
df_pbutton_16_17.replace(',','', regex=True, inplace=True)

,County,Buttonbuck_2016-17,Female_or_Antlerless_2016-17,Total_2016-17
1,Allegany,174,1071,1245
2,Anne Arundel,310,1580,1890
3,Baltimore,425,3314,3739
4,Calvert,193,1060,1253
5,Caroline,,,
6,Whitetail,338,1695,2033
7,Sika,0,1,1
8,Carroll,505,3156,3661
9,Cecil,342,1969,2311
10,Charles,257,1743,2000


In [90]:
#--------2016-2017 Button Buck and Doe Harvest Cleaning Pt. 3--------#

#7. Change datatype from string to float for numeric avlues 

df_pbutton_16_17['Buttonbuck_2016-17'] = pd.to_numeric( df_pbutton_16_17['Buttonbuck_2016-17'], errors="coerce")
df_pbutton_16_17['Female_or_Antlerless_2016-17'] = pd.to_numeric( df_pbutton_16_17['Female_or_Antlerless_2016-17'], errors="coerce")
df_pbutton_16_17['Total_2016-17'] = pd.to_numeric( df_pbutton_16_17['Total_2016-17'], errors="coerce")

In [91]:
#--------2016-2017 Button Buck and Doe Harvest Cleaning Pt. 4--------#

#8. Reset index after dropping rows. 
df_pbutton_16_17 = df_pbutton_16_17.reset_index(drop=True) 

#9. Sum whitetail and sika, then move those sums to county row 

#CAROLINE COUNTY
df_pbutton_16_17.iloc[4, 1] = df_pbutton_16_17.iloc[[5, 6], 1].sum()
df_pbutton_16_17.iloc[4, 2] = df_pbutton_16_17.iloc[[5,6], 2].sum()
df_pbutton_16_17.iloc[4, 3] = df_pbutton_16_17.iloc[[5, 6], 3].sum()

# #Dorchester
df_pbutton_16_17.iloc[10, 1] = df_pbutton_16_17.iloc[[11, 12], 1].sum()
df_pbutton_16_17.iloc[10, 2] = df_pbutton_16_17.iloc[[11,12], 2].sum()
df_pbutton_16_17.iloc[10, 3] = df_pbutton_16_17.iloc[[11, 12], 3].sum()

#Somerset
df_pbutton_16_17.iloc[22, 1] = df_pbutton_16_17.iloc[[23, 24], 1].sum()
df_pbutton_16_17.iloc[22, 2] = df_pbutton_16_17.iloc[[23,24], 2].sum()
df_pbutton_16_17.iloc[22, 3] = df_pbutton_16_17.iloc[[23, 24], 3].sum()

#Wicomico
df_pbutton_16_17.iloc[27, 1] = df_pbutton_16_17.iloc[[28, 29], 1].sum()
df_pbutton_16_17.iloc[27, 2] = df_pbutton_16_17.iloc[[28,29], 2].sum()
df_pbutton_16_17.iloc[27, 3] = df_pbutton_16_17.iloc[[28, 29], 3].sum()

# #Worcester
df_pbutton_16_17.iloc[30, 1] = df_pbutton_16_17.iloc[[31, 32], 1].sum()
df_pbutton_16_17.iloc[30, 2] = df_pbutton_16_17.iloc[[31,32], 2].sum()
df_pbutton_16_17.iloc[30, 3] = df_pbutton_16_17.iloc[[31, 32], 3].sum()

In [92]:
#---------2016-2017 Button Buck and Doe Harvest FINAL CLEANING: DROPPING ROWS---------#
#Filter county values for "whitetail" and "sika" and drop those rows
df_pbutton_16_17=df_pbutton_16_17[ df_pbutton_16_17[ "County" ].str.contains( "Whitetail|Sika" )== False]
df_pbutton_16_17= df_pbutton_16_17.reset_index(drop=True)
df_pbutton_16_17.to_csv("df_pbutton_16_17.csv")

In [93]:
#2017-2018 Button buck and doe harvest 

# Open a PDF
pdf = PDF('MD-Annual-Deer-Report-2017-2018.pdf')
page = pdf.pages[11]
page.show(width=700)

table = page.extract_table(method="pdfplumber")
list(table)

rows = list(table)

df_pbutton_17_18 = pd.DataFrame(rows)

df_pbutton_17_18.to_csv("df_pbutton_17_18.csv", index=False)

In [94]:
df_pbutton_17_18

,0,1,2,3,4
0,COUNTY,Buttonbuck,Female or\nAntlerless,Total,Percent\nButtonbuck
1,Allegany,146,1137,1283,11.4
2,Anne\nArundel,300,1701,2001,15.0
3,Baltimore,470,3335,3805,12.4
4,Calvert,166,964,1130,14.7
5,Caroline,,,,
6,Whitetail,327,1497,1824,17.9
7,Sika,1,0,1,*
8,Carroll,480,3300,3780,12.7
9,Cecil,347,2095,2442,14.2


In [95]:
#--------2017-2018 Button Buck and Doe Harvest Cleaning Pt. 1--------#

#1. RENAME COLUMNS BY INDEX
#https://www.geeksforgeeks.org/python/rename-column-by-index-in-pandas/

df_pbutton_17_18.rename(columns={
    df_pbutton_17_18.columns[0]: "County",
    df_pbutton_17_18.columns[1]: "Buttonbuck_2017-18",
    df_pbutton_17_18.columns[2]: "Female_or_Antlerless_2017-18",
    df_pbutton_17_18.columns[3]: "Total_2017-18",
    df_pbutton_17_18.columns[4]: "Percent_Buttonbuck"
    
}, inplace=True)
 
#2. FORMAT OF COUNTY NAMES:
    #strip \n
    #Anne Arundel
    #Prince George's 
    #Queen Anne's
df_pbutton_17_18['County']=df_pbutton_17_18['County'].str.replace("\n", " ")
#https://stackoverflow.com/questions/65666852/why-cant-i-replace-a-newline-in-my-pandas-dataframe
#https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

#3. Drop percent in buttonbuck
df_pbutton_17_18 = df_pbutton_17_18.drop(df_pbutton_17_18.columns[4], axis=1)

In [96]:
df_pbutton_17_18

,County,Buttonbuck_2017-18,Female_or_Antlerless_2017-18,Total_2017-18
0,COUNTY,Buttonbuck,Female or\nAntlerless,Total
1,Allegany,146,1137,1283
2,Anne Arundel,300,1701,2001
3,Baltimore,470,3335,3805
4,Calvert,166,964,1130
5,Caroline,,,
6,Whitetail,327,1497,1824
7,Sika,1,0,1
8,Carroll,480,3300,3780
9,Cecil,347,2095,2442


In [97]:
#--------2017-2018 Button Buck and Doe Harvest Cleaning Pt. 2--------#

#4. Replace asterisks and spaces with NaN
df_pbutton_17_18=df_pbutton_17_18.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

#5. Drop last row, which just has "small sample size" text
df_pbutton_17_18 = df_pbutton_17_18.iloc[:-1]

#6. Drop first row
df_pbutton_17_18.drop(labels=0, inplace=True)

#7. Strip commas
df_pbutton_17_18.replace(',','', regex=True, inplace=True)

,County,Buttonbuck_2017-18,Female_or_Antlerless_2017-18,Total_2017-18
1,Allegany,146,1137,1283
2,Anne Arundel,300,1701,2001
3,Baltimore,470,3335,3805
4,Calvert,166,964,1130
5,Caroline,,,
6,Whitetail,327,1497,1824
7,Sika,1,0,1
8,Carroll,480,3300,3780
9,Cecil,347,2095,2442
10,Charles,304,1942,2246


In [98]:
#--------2017-2018 Button Buck and Doe Harvest Cleaning Pt. 3--------#

#7. Change datatype from string to float for numeric values 

df_pbutton_17_18['Buttonbuck_2017-18'] = pd.to_numeric( df_pbutton_17_18['Buttonbuck_2017-18'], errors="coerce")
df_pbutton_17_18['Female_or_Antlerless_2017-18'] = pd.to_numeric( df_pbutton_17_18['Female_or_Antlerless_2017-18'], errors="coerce")
df_pbutton_17_18['Total_2017-18'] = pd.to_numeric( df_pbutton_17_18['Total_2017-18'], errors="coerce")

#8. Reset index after dropping rows. 
df_pbutton_17_18 = df_pbutton_17_18.reset_index(drop=True) 

In [99]:
df_pbutton_17_18

,County,Buttonbuck_2017-18,Female_or_Antlerless_2017-18,Total_2017-18
0,Allegany,146.0,1137.0,1283.0
1,Anne Arundel,300.0,1701.0,2001.0
2,Baltimore,470.0,3335.0,3805.0
3,Calvert,166.0,964.0,1130.0
4,Caroline,NaN,NaN,NaN
5,Whitetail,327.0,1497.0,1824.0
6,Sika,1.0,0.0,1.0
7,Carroll,480.0,3300.0,3780.0
8,Cecil,347.0,2095.0,2442.0
9,Charles,304.0,1942.0,2246.0


In [100]:
#--------2017-2018 Button Buck and Doe Harvest Cleaning Pt. 4--------# 

#9. Sum whitetail and sika, then move those sums to county row 

#CAROLINE COUNTY
df_pbutton_17_18.iloc[4, 1] = df_pbutton_17_18.iloc[[5, 6], 1].sum()
df_pbutton_17_18.iloc[4, 2] = df_pbutton_17_18.iloc[[5,6], 2].sum()
df_pbutton_17_18.iloc[4, 3] = df_pbutton_17_18.iloc[[5, 6], 3].sum()

# #Dorchester
df_pbutton_17_18.iloc[10, 1] = df_pbutton_17_18.iloc[[11, 12], 1].sum()
df_pbutton_17_18.iloc[10, 2] = df_pbutton_17_18.iloc[[11,12], 2].sum()
df_pbutton_17_18.iloc[10, 3] = df_pbutton_17_18.iloc[[11, 12], 3].sum()

#Somerset
df_pbutton_17_18.iloc[22, 1] = df_pbutton_17_18.iloc[[23, 24], 1].sum()
df_pbutton_17_18.iloc[22, 2] = df_pbutton_17_18.iloc[[23,24], 2].sum()
df_pbutton_17_18.iloc[22, 3] = df_pbutton_17_18.iloc[[23, 24], 3].sum()

#Wicomico
df_pbutton_17_18.iloc[27, 1] = df_pbutton_17_18.iloc[[28, 29], 1].sum()
df_pbutton_17_18.iloc[27, 2] = df_pbutton_17_18.iloc[[28,29], 2].sum()
df_pbutton_17_18.iloc[27, 3] = df_pbutton_17_18.iloc[[28, 29], 3].sum()

# #Worcester
df_pbutton_17_18.iloc[30, 1] = df_pbutton_17_18.iloc[[31, 32], 1].sum()
df_pbutton_17_18.iloc[30, 2] = df_pbutton_17_18.iloc[[31,32], 2].sum()
df_pbutton_17_18.iloc[30, 3] = df_pbutton_17_18.iloc[[31, 32], 3].sum()

In [101]:
#---------2017-2018 Button Buck and Doe Harvest FINAL CLEANING: DROPPING ROWS---------#
#10. Filter county values for "whitetail" and "sika" and drop those rows
df_pbutton_17_18=df_pbutton_17_18[ df_pbutton_17_18[ "County" ].str.contains( "Whitetail|Sika" )== False]
df_pbutton_17_18= df_pbutton_17_18.reset_index(drop=True)
df_pbutton_17_18.to_csv("df_pbutton_17_18.csv")

In [102]:
#2018-2019 Button buck and doe harvest 
from natural_pdf import PDF
# Open a PDF
pdf = PDF('MD-Annual-Deer-Report-2018-2019.pdf')
page = pdf.pages[13]
page.show(width=700)

table = page.extract_table(method="pdfplumber")
list(table)

rows = list(table)

df_pbutton_18_19 = pd.DataFrame(rows)

df_pbutton_18_19.to_csv("df_pbutton_18_19.csv", index=False)

In [103]:
#--------2018-2019 Button Buck and Doe Harvest Cleaning Pt. 1--------#

#1. RENAME COLUMNS BY INDEX
#https://www.geeksforgeeks.org/python/rename-column-by-index-in-pandas/

df_pbutton_18_19.rename(columns={
    df_pbutton_18_19.columns[0]: "County",
    df_pbutton_18_19.columns[1]: "Buttonbuck_2018-19",
    df_pbutton_18_19.columns[2]: "Female_or_Antlerless_2018-19",
    df_pbutton_18_19.columns[3]: "Total_2018-19",
    df_pbutton_18_19.columns[4]: "Percent_Buttonbuck"
    
}, inplace=True)
 
#2. FORMAT OF COUNTY NAMES:
    #strip \n
    #Anne Arundel
    #Prince George's 
    #Queen Anne's
df_pbutton_18_19['County']=df_pbutton_18_19['County'].str.replace("\n", " ")
#https://stackoverflow.com/questions/65666852/why-cant-i-replace-a-newline-in-my-pandas-dataframe
#https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

#3. Drop percent in buttonbuck
df_pbutton_18_19 = df_pbutton_18_19.drop(df_pbutton_18_19.columns[4], axis=1)

In [104]:
df_pbutton_18_19

,County,Buttonbuck_2018-19,Female_or_Antlerless_2018-19,Total_2018-19
0,COUNTY,Buttonbuck,Female or\nAntlerless,Total
1,Allegany,156,"1,288","1,444"
2,Anne Arundel,219,"1,343","1,562"
3,Baltimore,330,"2,633","2,963"
4,Calvert,130,817,947
5,Caroline,,,
6,Whitetail,257,"1,470","1,727"
7,Sika,0,1,1
8,Carroll,384,"2,840","3,224"
9,Cecil,290,"1,979","2,269"


In [105]:
#--------2018-2019 Button Buck and Doe Harvest Cleaning Pt. 2--------#
import numpy as np
#4. Replace asterisks and spaces with NaN
df_pbutton_18_19=df_pbutton_18_19.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

#5. Drop last row, which just has "small sample size" text
df_pbutton_18_19 = df_pbutton_18_19.iloc[:-1]

#6. Drop first row
df_pbutton_18_19.drop(labels=0, inplace=True)

#7. Strip commas
df_pbutton_18_19.replace(',','', regex=True, inplace=True)

,County,Buttonbuck_2018-19,Female_or_Antlerless_2018-19,Total_2018-19
1,Allegany,156,1288,1444
2,Anne Arundel,219,1343,1562
3,Baltimore,330,2633,2963
4,Calvert,130,817,947
5,Caroline,,,
6,Whitetail,257,1470,1727
7,Sika,0,1,1
8,Carroll,384,2840,3224
9,Cecil,290,1979,2269
10,Charles,155,1173,1328


In [106]:
#--------2018-2019 Button Buck and Doe Harvest Cleaning Pt. 3--------#

#8. Change datatype from string to float for numeric values 

df_pbutton_18_19['Buttonbuck_2018-19'] = pd.to_numeric( df_pbutton_18_19['Buttonbuck_2018-19'], errors="coerce")
df_pbutton_18_19['Female_or_Antlerless_2018-19'] = pd.to_numeric( df_pbutton_18_19['Female_or_Antlerless_2018-19'], errors="coerce")
df_pbutton_18_19['Total_2018-19'] = pd.to_numeric( df_pbutton_18_19['Total_2018-19'], errors="coerce")

#9. Reset index after dropping rows. 
df_pbutton_18_19 = df_pbutton_18_19.reset_index(drop=True) 

In [107]:
df_pbutton_18_19

,County,Buttonbuck_2018-19,Female_or_Antlerless_2018-19,Total_2018-19
0,Allegany,156.0,1288.0,1444.0
1,Anne Arundel,219.0,1343.0,1562.0
2,Baltimore,330.0,2633.0,2963.0
3,Calvert,130.0,817.0,947.0
4,Caroline,NaN,NaN,NaN
5,Whitetail,257.0,1470.0,1727.0
6,Sika,0.0,1.0,1.0
7,Carroll,384.0,2840.0,3224.0
8,Cecil,290.0,1979.0,2269.0
9,Charles,155.0,1173.0,1328.0


In [108]:
#--------2018-2019 Button Buck and Doe Harvest Cleaning Pt. 4--------#

#10. Sum whitetail and sika, then move those sums to county row 

#CAROLINE COUNTY
df_pbutton_18_19.iloc[4, 1] = df_pbutton_18_19.iloc[[5, 6], 1].sum()
df_pbutton_18_19.iloc[4, 2] = df_pbutton_18_19.iloc[[5,6], 2].sum()
df_pbutton_18_19.iloc[4, 3] = df_pbutton_18_19.iloc[[5, 6], 3].sum()

# #Dorchester
df_pbutton_18_19.iloc[10, 1] = df_pbutton_18_19.iloc[[11, 12], 1].sum()
df_pbutton_18_19.iloc[10, 2] = df_pbutton_18_19.iloc[[11,12], 2].sum()
df_pbutton_18_19.iloc[10, 3] = df_pbutton_18_19.iloc[[11, 12], 3].sum()

#Somerset
df_pbutton_18_19.iloc[22, 1] = df_pbutton_18_19.iloc[[23, 24], 1].sum()
df_pbutton_18_19.iloc[22, 2] = df_pbutton_18_19.iloc[[23,24], 2].sum()
df_pbutton_18_19.iloc[22, 3] = df_pbutton_18_19.iloc[[23, 24], 3].sum()

#Wicomico
df_pbutton_18_19.iloc[27, 1] = df_pbutton_18_19.iloc[[28, 29], 1].sum()
df_pbutton_18_19.iloc[27, 2] = df_pbutton_18_19.iloc[[28,29], 2].sum()
df_pbutton_18_19.iloc[27, 3] = df_pbutton_18_19.iloc[[28, 29], 3].sum()

# #Worcester
df_pbutton_18_19.iloc[30, 1] = df_pbutton_18_19.iloc[[31, 32], 1].sum()
df_pbutton_18_19.iloc[30, 2] = df_pbutton_18_19.iloc[[31,32], 2].sum()
df_pbutton_18_19.iloc[30, 3] = df_pbutton_18_19.iloc[[31, 32], 3].sum()

In [109]:
df_pbutton_18_19

,County,Buttonbuck_2018-19,Female_or_Antlerless_2018-19,Total_2018-19
0,Allegany,156.0,1288.0,1444.0
1,Anne Arundel,219.0,1343.0,1562.0
2,Baltimore,330.0,2633.0,2963.0
3,Calvert,130.0,817.0,947.0
4,Caroline,257.0,1471.0,1728.0
5,Whitetail,257.0,1470.0,1727.0
6,Sika,0.0,1.0,1.0
7,Carroll,384.0,2840.0,3224.0
8,Cecil,290.0,1979.0,2269.0
9,Charles,155.0,1173.0,1328.0


In [110]:
#---------2018-2019 Button Buck and Doe Harvest FINAL CLEANING: DROPPING ROWS---------#
#11. Filter county values for "whitetail" and "sika" and drop those rows
df_pbutton_18_19=df_pbutton_18_19[ df_pbutton_18_19[ "County" ].str.contains( "Whitetail|Sika" )== False]
df_pbutton_18_19= df_pbutton_18_19.reset_index(drop=True)
df_pbutton_18_19.to_csv("df_pbutton_18_19.csv")

In [111]:
#2019-2020 Button buck and doe harvest 

# Open a PDF
pdf = PDF('MD-Annual-Deer-Report-2019-2020.pdf')
page = pdf.pages[5]
page.show(width=700)

table = page.extract_table(method="pdfplumber")
list(table)

rows = list(table)

df_pbutton_19_20 = pd.DataFrame(rows)

df_pbutton_19_20.to_csv("df_pbutton_19_20.csv", index=False)

In [112]:
#--------2019-2020 Button Buck and Doe Harvest Cleaning Pt. 1--------#

#1. RENAME COLUMNS BY INDEX
#https://www.geeksforgeeks.org/python/rename-column-by-index-in-pandas/

df_pbutton_19_20.rename(columns={
    df_pbutton_19_20.columns[0]: "County",
    df_pbutton_19_20.columns[1]: "Buttonbuck_2019-20",
    df_pbutton_19_20.columns[2]: "Female_or_Antlerless_2019-20",
    df_pbutton_19_20.columns[3]: "Total_2019-20",
    df_pbutton_19_20.columns[4]: "Percent_Buttonbuck"
    
}, inplace=True)
 
#2. FORMAT OF COUNTY NAMES:
    #strip \n
    #Anne Arundel
    #Prince George's 
    #Queen Anne's
df_pbutton_19_20['County']=df_pbutton_19_20['County'].str.replace("\n", " ")
#https://stackoverflow.com/questions/65666852/why-cant-i-replace-a-newline-in-my-pandas-dataframe
#https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

#3. Drop percent in buttonbuck
df_pbutton_19_20 = df_pbutton_19_20.drop(df_pbutton_19_20.columns[4], axis=1)

In [113]:
#--------2019-2020 Button Buck and Doe Harvest Cleaning Pt. 2--------#

#4. Replace asterisks and spaces with NaN
df_pbutton_19_20=df_pbutton_19_20.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

#5. Drop last row, which just has "small sample size" text
df_pbutton_19_20 = df_pbutton_19_20.iloc[:-1]

#6. Drop first row
df_pbutton_19_20.drop(labels=0, inplace=True)

#7. Strip commas
df_pbutton_19_20.replace(',','', regex=True, inplace=True)

,County,Buttonbuck_2019-20,Female_or_Antlerless_2019-20,Total_2019-20
1,Allegany,91,1088,1179
2,Anne Arundel,199,1454,1653
3,Baltimore,386,2809,3195
4,Calvert,148,877,1025
5,Caroline,,,
6,Whitetail,271,1526,1797
7,Sika,0,1,1
8,Carroll,423,3114,3537
9,Cecil,333,2135,2468
10,Charles,257,1611,1868


In [114]:
#--------2019-2020 Button Buck and Doe Harvest Cleaning Pt. 3--------#

#8. Change datatype from string to float for numeric avlues 

df_pbutton_19_20['Buttonbuck_2019-20'] = pd.to_numeric( df_pbutton_19_20['Buttonbuck_2019-20'], errors="coerce")
df_pbutton_19_20['Female_or_Antlerless_2019-20'] = pd.to_numeric( df_pbutton_19_20['Female_or_Antlerless_2019-20'], errors="coerce")
df_pbutton_19_20['Total_2019-20'] = pd.to_numeric( df_pbutton_19_20['Total_2019-20'], errors="coerce")

#9. Reset index after dropping rows. 
df_pbutton_19_20 = df_pbutton_19_20.reset_index(drop=True) 

In [115]:
df_pbutton_19_20

,County,Buttonbuck_2019-20,Female_or_Antlerless_2019-20,Total_2019-20
0,Allegany,91.0,1088.0,1179.0
1,Anne Arundel,199.0,1454.0,1653.0
2,Baltimore,386.0,2809.0,3195.0
3,Calvert,148.0,877.0,1025.0
4,Caroline,NaN,NaN,NaN
5,Whitetail,271.0,1526.0,1797.0
6,Sika,0.0,1.0,1.0
7,Carroll,423.0,3114.0,3537.0
8,Cecil,333.0,2135.0,2468.0
9,Charles,257.0,1611.0,1868.0


In [116]:
#--------2019-2020 Button Buck and Doe Harvest Cleaning Pt. 4--------#

#10. Sum whitetail and sika, then move those sums to county row 

#CAROLINE COUNTY
df_pbutton_19_20.iloc[4, 1] = df_pbutton_19_20.iloc[[5, 6], 1].sum()
df_pbutton_19_20.iloc[4, 2] = df_pbutton_19_20.iloc[[5,6], 2].sum()
df_pbutton_19_20.iloc[4, 3] = df_pbutton_19_20.iloc[[5, 6], 3].sum()

# #Dorchester
df_pbutton_19_20.iloc[10, 1] = df_pbutton_19_20.iloc[[11, 12], 1].sum()
df_pbutton_19_20.iloc[10, 2] = df_pbutton_19_20.iloc[[11,12], 2].sum()
df_pbutton_19_20.iloc[10, 3] = df_pbutton_19_20.iloc[[11, 12], 3].sum()

#Somerset
df_pbutton_19_20.iloc[22, 1] = df_pbutton_19_20.iloc[[23, 24], 1].sum()
df_pbutton_19_20.iloc[22, 2] = df_pbutton_19_20.iloc[[23,24], 2].sum()
df_pbutton_19_20.iloc[22, 3] = df_pbutton_19_20.iloc[[23, 24], 3].sum()

#Wicomico
df_pbutton_19_20.iloc[27, 1] = df_pbutton_19_20.iloc[[28, 29], 1].sum()
df_pbutton_19_20.iloc[27, 2] = df_pbutton_19_20.iloc[[28,29], 2].sum()
df_pbutton_19_20.iloc[27, 3] = df_pbutton_19_20.iloc[[28, 29], 3].sum()

# #Worcester
df_pbutton_19_20.iloc[30, 1] = df_pbutton_19_20.iloc[[31, 32], 1].sum()
df_pbutton_19_20.iloc[30, 2] = df_pbutton_19_20.iloc[[31,32], 2].sum()
df_pbutton_19_20.iloc[30, 3] = df_pbutton_19_20.iloc[[31, 32], 3].sum()

In [117]:
#---------2019-2020 Button Buck and Doe Harvest FINAL CLEANING: DROPPING ROWS---------#
#11. Filter county values for "whitetail" and "sika" and drop those rows
df_pbutton_19_20=df_pbutton_19_20[ df_pbutton_19_20[ "County" ].str.contains( "Whitetail|Sika" )== False]
df_pbutton_19_20= df_pbutton_19_20.reset_index(drop=True)
df_pbutton_19_20.to_csv("df_pbutton_19_20.csv")

In [118]:
#2020-2021 Button buck and doe harvest 

# Open a PDF
pdf = PDF('Maryland-Annual-Deer-Report-2020-2021.pdf')
page = pdf.pages[6]
page.show(width=700)

table = page.extract_table(method="pdfplumber")
list(table)

rows = list(table)

df_pbutton_20_21 = pd.DataFrame(rows)

df_pbutton_20_21.to_csv("df_pbutton_20_21.csv", index=False)

In [119]:
#--------2020-2021 Button Buck and Doe Harvest Cleaning Pt. 1--------#

#1. RENAME COLUMNS BY INDEX
#https://www.geeksforgeeks.org/python/rename-column-by-index-in-pandas/

df_pbutton_20_21.rename(columns={
    df_pbutton_20_21.columns[0]: "County",
    df_pbutton_20_21.columns[1]: "Buttonbuck_2020-21",
    df_pbutton_20_21.columns[2]: "Female_or_Antlerless_2020-21",
    df_pbutton_20_21.columns[3]: "Total_2020-21",
    df_pbutton_20_21.columns[4]: "Percent_Buttonbuck"
    
}, inplace=True)
 
#2. FORMAT OF COUNTY NAMES:
    #strip \n
    #Anne Arundel
    #Prince George's 
    #Queen Anne's
df_pbutton_20_21['County']=df_pbutton_20_21['County'].str.replace("\n", " ")
#https://stackoverflow.com/questions/65666852/why-cant-i-replace-a-newline-in-my-pandas-dataframe
#https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

#3. Drop percent in buttonbuck
df_pbutton_20_21 = df_pbutton_20_21.drop(df_pbutton_20_21.columns[4], axis=1)

In [120]:
#--------2020-2021 Button Buck and Doe Harvest Cleaning Pt. 2--------#

#4. Replace asterisks and spaces with NaN
df_pbutton_20_21=df_pbutton_20_21.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

#5. Drop last row, which just has "small sample size" text
df_pbutton_20_21 = df_pbutton_20_21.iloc[:-1]

#6. Drop first row
df_pbutton_20_21.drop(labels=0, inplace=True)

#7. Strip commas
df_pbutton_20_21.replace(',','', regex=True, inplace=True)

,County,Buttonbuck_2020-21,Female_or_Antlerless_2020-21,Total_2020-21
1,Allegany,124,1255,1379
2,Anne Arundel,181,1096,1277
3,Baltimore,410,3165,3575
4,Calvert,131,785,916
5,Caroline,,,
6,Whitetail,295,1599,1894
7,Sika,1,1,2
8,Carroll,515,3491,4006
9,Cecil,327,2203,2530
10,Charles,198,1359,1557


In [121]:
#--------2020-2021 Button Buck and Doe Harvest Cleaning Pt. 3--------#

#8. Change datatype from string to float for numeric avlues 

df_pbutton_20_21['Buttonbuck_2020-21'] = pd.to_numeric( df_pbutton_20_21['Buttonbuck_2020-21'], errors="coerce")
df_pbutton_20_21['Female_or_Antlerless_2020-21'] = pd.to_numeric( df_pbutton_20_21['Female_or_Antlerless_2020-21'], errors="coerce")
df_pbutton_20_21['Total_2020-21'] = pd.to_numeric( df_pbutton_20_21['Total_2020-21'], errors="coerce")

#9. Reset index after dropping rows. 
df_pbutton_20_21 = df_pbutton_20_21.reset_index(drop=True) 

In [122]:
df_pbutton_20_21

,County,Buttonbuck_2020-21,Female_or_Antlerless_2020-21,Total_2020-21
0,Allegany,124.0,1255.0,1379.0
1,Anne Arundel,181.0,1096.0,1277.0
2,Baltimore,410.0,3165.0,3575.0
3,Calvert,131.0,785.0,916.0
4,Caroline,NaN,NaN,NaN
5,Whitetail,295.0,1599.0,1894.0
6,Sika,1.0,1.0,2.0
7,Carroll,515.0,3491.0,4006.0
8,Cecil,327.0,2203.0,2530.0
9,Charles,198.0,1359.0,1557.0


In [123]:
#--------2020-2021 Button Buck and Doe Harvest Cleaning Pt. 4--------#

#10. Sum whitetail and sika, then move those sums to county row 

#CAROLINE COUNTY
df_pbutton_20_21.iloc[4, 1] = df_pbutton_20_21.iloc[[5, 6], 1].sum()
df_pbutton_20_21.iloc[4, 2] = df_pbutton_20_21.iloc[[5,6], 2].sum()
df_pbutton_20_21.iloc[4, 3] = df_pbutton_20_21.iloc[[5, 6], 3].sum()

# #Dorchester
df_pbutton_20_21.iloc[10, 1] = df_pbutton_20_21.iloc[[11, 12], 1].sum()
df_pbutton_20_21.iloc[10, 2] = df_pbutton_20_21.iloc[[11,12], 2].sum()
df_pbutton_20_21.iloc[10, 3] = df_pbutton_20_21.iloc[[11, 12], 3].sum()

#Somerset
df_pbutton_20_21.iloc[22, 1] = df_pbutton_20_21.iloc[[23, 24], 1].sum()
df_pbutton_20_21.iloc[22, 2] = df_pbutton_20_21.iloc[[23,24], 2].sum()
df_pbutton_20_21.iloc[22, 3] = df_pbutton_20_21.iloc[[23, 24], 3].sum()

#Wicomico
df_pbutton_20_21.iloc[27, 1] = df_pbutton_20_21.iloc[[28, 29], 1].sum()
df_pbutton_20_21.iloc[27, 2] = df_pbutton_20_21.iloc[[28,29], 2].sum()
df_pbutton_20_21.iloc[27, 3] = df_pbutton_20_21.iloc[[28, 29], 3].sum()

# #Worcester
df_pbutton_20_21.iloc[30, 1] = df_pbutton_20_21.iloc[[31, 32], 1].sum()
df_pbutton_20_21.iloc[30, 2] = df_pbutton_20_21.iloc[[31,32], 2].sum()
df_pbutton_20_21.iloc[30, 3] = df_pbutton_20_21.iloc[[31, 32], 3].sum()

In [124]:
#---------2020-2021 Button Buck and Doe Harvest FINAL CLEANING: DROPPING ROWS---------#
#11. Filter county values for "whitetail" and "sika" and drop those rows
df_pbutton_20_21=df_pbutton_20_21[ df_pbutton_20_21[ "County" ].str.contains( "Whitetail|Sika" )== False]
df_pbutton_20_21= df_pbutton_20_21.reset_index(drop=True)
df_pbutton_20_21.to_csv("df_pbutton_20_21.csv")

In [125]:
#2021-2022 Button buck and doe harvest 

# Open a PDF
pdf = PDF('Maryland-Big-Game-Report-2021-22.pdf')
page = pdf.pages[7]
page.show(width=700)

table = page.extract_table(method="pdfplumber")
list(table)

rows = list(table)

df_pbutton_21_22 = pd.DataFrame(rows)

df_pbutton_21_22.to_csv("df_pbutton_21_22.csv", index=False)

In [126]:
#--------2021-2022 Button Buck and Doe Harvest Cleaning Pt. 1--------#

#1. RENAME COLUMNS BY INDEX
#https://www.geeksforgeeks.org/python/rename-column-by-index-in-pandas/

df_pbutton_21_22.rename(columns={
    df_pbutton_21_22.columns[0]: "County",
    df_pbutton_21_22.columns[1]: "Buttonbuck_2021-22",
    df_pbutton_21_22.columns[2]: "Female_or_Antlerless_2021-22",
    df_pbutton_21_22.columns[3]: "Total_2021-22",
    df_pbutton_21_22.columns[4]: "Percent_Buttonbuck"
    
}, inplace=True)
 
#2. FORMAT OF COUNTY NAMES:
    #strip \n
    #Anne Arundel
    #Prince George's 
    #Queen Anne's
df_pbutton_21_22['County']=df_pbutton_21_22['County'].str.replace("\n", " ")
#https://stackoverflow.com/questions/65666852/why-cant-i-replace-a-newline-in-my-pandas-dataframe
#https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

#3. Drop percent in buttonbuck
df_pbutton_21_22 = df_pbutton_21_22.drop(df_pbutton_21_22.columns[4], axis=1)

In [127]:
#--------2021-2022 Button Buck and Doe Harvest Cleaning Pt. 2--------#

#4. Replace asterisks and spaces with NaN
df_pbutton_21_22=df_pbutton_21_22.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

#5. Drop last row, which just has "small sample size" text
df_pbutton_21_22 = df_pbutton_21_22.iloc[:-1]

#6. Drop first row
df_pbutton_21_22.drop(labels=0, inplace=True)

#7. Strip commas
df_pbutton_21_22.replace(',','', regex=True, inplace=True)

,County,Buttonbuck_2021-22,Female_or_Antlerless_2021-22,Total_2021-22
1,Allegany,106,1067,1173
2,Anne Arundel,152,930,1082
3,Baltimore,321,2500,2821
4,Calvert,109,678,787
5,Caroline,253,1323,1576
6,Carroll,362,2651,3013
7,Cecil,259,1737,1996
8,Charles,140,942,1082
9,Dorchester,,,
10,Whitetail,232,1003,1235


In [128]:
#--------2021-2022 Button Buck and Doe Harvest Cleaning Pt. 3--------#

#8. Change datatype from string to float for numeric avlues 

df_pbutton_21_22['Buttonbuck_2021-22'] = pd.to_numeric( df_pbutton_21_22['Buttonbuck_2021-22'], errors="coerce")
df_pbutton_21_22['Female_or_Antlerless_2021-22'] = pd.to_numeric( df_pbutton_21_22['Female_or_Antlerless_2021-22'], errors="coerce")
df_pbutton_21_22['Total_2021-22'] = pd.to_numeric( df_pbutton_21_22['Total_2021-22'], errors="coerce")

#9. Reset index after dropping rows. 
df_pbutton_21_22 = df_pbutton_21_22.reset_index(drop=True) 

In [129]:
df_pbutton_21_22

,County,Buttonbuck_2021-22,Female_or_Antlerless_2021-22,Total_2021-22
0,Allegany,106.0,1067.0,1173.0
1,Anne Arundel,152.0,930.0,1082.0
2,Baltimore,321.0,2500.0,2821.0
3,Calvert,109.0,678.0,787.0
4,Caroline,253.0,1323.0,1576.0
5,Carroll,362.0,2651.0,3013.0
6,Cecil,259.0,1737.0,1996.0
7,Charles,140.0,942.0,1082.0
8,Dorchester,NaN,NaN,NaN
9,Whitetail,232.0,1003.0,1235.0


In [130]:
#--------2021-2022 Button Buck and Doe Harvest Cleaning Pt. 4--------#

#10. Sum whitetail and sika, then move those sums to county row 
#CAROLINE COUNTY DIDN'T HAVE WHITETAIL AND SIKA DISTINCTION IN 2021-2022 REPORT 

# #Dorchester
df_pbutton_21_22.iloc[8, 1] = df_pbutton_21_22.iloc[[9, 10], 1].sum()
df_pbutton_21_22.iloc[8, 2] = df_pbutton_21_22.iloc[[9,10], 2].sum()
df_pbutton_21_22.iloc[8, 3] = df_pbutton_21_22.iloc[[9, 10], 3].sum()

#Somerset
df_pbutton_21_22.iloc[20, 1] = df_pbutton_21_22.iloc[[21, 22], 1].sum()
df_pbutton_21_22.iloc[20, 2] = df_pbutton_21_22.iloc[[21,22], 2].sum()
df_pbutton_21_22.iloc[20, 3] = df_pbutton_21_22.iloc[[21, 22], 3].sum()

#Wicomico
df_pbutton_21_22.iloc[25, 1] = df_pbutton_21_22.iloc[[26, 27], 1].sum()
df_pbutton_21_22.iloc[25, 2] = df_pbutton_21_22.iloc[[26,27], 2].sum()
df_pbutton_21_22.iloc[25, 3] = df_pbutton_21_22.iloc[[26, 27], 3].sum()

# #Worcester
df_pbutton_21_22.iloc[28, 1] = df_pbutton_21_22.iloc[[29, 30], 1].sum()
df_pbutton_21_22.iloc[28, 2] = df_pbutton_21_22.iloc[[29,30], 2].sum()
df_pbutton_21_22.iloc[28, 3] = df_pbutton_21_22.iloc[[29, 30], 3].sum()

In [131]:
df_pbutton_21_22

,County,Buttonbuck_2021-22,Female_or_Antlerless_2021-22,Total_2021-22
0,Allegany,106.0,1067.0,1173.0
1,Anne Arundel,152.0,930.0,1082.0
2,Baltimore,321.0,2500.0,2821.0
3,Calvert,109.0,678.0,787.0
4,Caroline,253.0,1323.0,1576.0
5,Carroll,362.0,2651.0,3013.0
6,Cecil,259.0,1737.0,1996.0
7,Charles,140.0,942.0,1082.0
8,Dorchester,329.0,2609.0,2938.0
9,Whitetail,232.0,1003.0,1235.0


In [132]:
#---------2021-2022 Button Buck and Doe Harvest FINAL CLEANING: DROPPING ROWS---------#
#11. Filter county values for "whitetail" and "sika" and drop those rows
df_pbutton_21_22=df_pbutton_21_22[ df_pbutton_21_22[ "County" ].str.contains( "Whitetail|Sika" )== False]
df_pbutton_21_22= df_pbutton_21_22.reset_index(drop=True)
df_pbutton_21_22.to_csv("df_pbutton_21_22.csv")

In [133]:
#2022-2023 Button buck and doe harvest 

# Open a PDF
pdf = PDF('Maryland-Big-Game-Report-2022-2023.pdf')
page = pdf.pages[7]
page.show(width=700)

table = page.extract_table(method="pdfplumber")
list(table)

rows = list(table)

df_pbutton_22_23 = pd.DataFrame(rows)

df_pbutton_22_23.to_csv("df_pbutton_22_23.csv", index=False)

In [134]:
#--------2022-2023 Button Buck and Doe Harvest Cleaning Pt. 1--------#

#1. RENAME COLUMNS BY INDEX
#https://www.geeksforgeeks.org/python/rename-column-by-index-in-pandas/

df_pbutton_22_23.rename(columns={
    df_pbutton_22_23.columns[0]: "County",
    df_pbutton_22_23.columns[1]: "Buttonbuck_2022-23",
    df_pbutton_22_23.columns[2]: "Female_or_Antlerless_2022-23",
    df_pbutton_22_23.columns[3]: "Total_2022-23",
    df_pbutton_22_23.columns[4]: "Percent_Buttonbuck"
    
}, inplace=True)
 
#2. FORMAT OF COUNTY NAMES:
    #strip \n
    #Anne Arundel
    #Prince George's 
    #Queen Anne's
df_pbutton_22_23['County']=df_pbutton_22_23['County'].str.replace("\n", " ")
#https://stackoverflow.com/questions/65666852/why-cant-i-replace-a-newline-in-my-pandas-dataframe
#https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

#3. Drop percent in buttonbuck
df_pbutton_22_23 = df_pbutton_22_23.drop(df_pbutton_22_23.columns[4], axis=1)

In [135]:
#--------2022-2023 Button Buck and Doe Harvest Cleaning Pt. 2--------#

#4. Replace asterisks and spaces with NaN
df_pbutton_22_23=df_pbutton_22_23.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

#5. Drop last row, which just has "small sample size" text
df_pbutton_22_23 = df_pbutton_22_23.iloc[:-1]

#6. Drop first row
df_pbutton_22_23.drop(labels=0, inplace=True)

#7. Strip commas
df_pbutton_22_23.replace(',','', regex=True, inplace=True)

,County,Buttonbuck_2022-23,Female_or_Antlerless_2022-23,Total_2022-23
1,Allegany,138,1336,1474
2,Anne Arundel,149,934,1083
3,Baltimore,353,2637,2990
4,Calvert,126,753,879
5,Caroline,,,
6,,315,1652,1967
7,,0,1,1
8,Carroll,433,3001,3434
9,Cecil,283,1839,2122
10,Charles,189,1250,1439


In [136]:
#--------2022-2023 Button Buck and Doe Harvest Cleaning Pt. 3--------#

#8. Change datatype from string to float for numeric avlues 

df_pbutton_22_23['Buttonbuck_2022-23'] = pd.to_numeric( df_pbutton_22_23['Buttonbuck_2022-23'], errors="coerce")
df_pbutton_22_23['Female_or_Antlerless_2022-23'] = pd.to_numeric( df_pbutton_22_23['Female_or_Antlerless_2022-23'], errors="coerce")
df_pbutton_22_23['Total_2022-23'] = pd.to_numeric( df_pbutton_22_23['Total_2022-23'], errors="coerce")

#9. Reset index after dropping rows. 
df_pbutton_22_23 = df_pbutton_22_23.reset_index(drop=True) 

In [137]:
df_pbutton_22_23

,County,Buttonbuck_2022-23,Female_or_Antlerless_2022-23,Total_2022-23
0,Allegany,138.0,1336.0,1474.0
1,Anne Arundel,149.0,934.0,1083.0
2,Baltimore,353.0,2637.0,2990.0
3,Calvert,126.0,753.0,879.0
4,Caroline,NaN,NaN,NaN
5,,315.0,1652.0,1967.0
6,,0.0,1.0,1.0
7,Carroll,433.0,3001.0,3434.0
8,Cecil,283.0,1839.0,2122.0
9,Charles,189.0,1250.0,1439.0


In [138]:
#--------2022-2023 Button Buck and Doe Harvest Cleaning Pt. 4--------#

#10. Sum whitetail and sika, then move those sums to county row 

#CAROLINE COUNTY
df_pbutton_22_23.iloc[4, 1] = df_pbutton_22_23.iloc[[5, 6], 1].sum()
df_pbutton_22_23.iloc[4, 2] = df_pbutton_22_23.iloc[[5,6], 2].sum()
df_pbutton_22_23.iloc[4, 3] = df_pbutton_22_23.iloc[[5, 6], 3].sum()

# #Dorchester
df_pbutton_22_23.iloc[10, 1] = df_pbutton_22_23.iloc[[11, 12], 1].sum()
df_pbutton_22_23.iloc[10, 2] = df_pbutton_22_23.iloc[[11,12], 2].sum()
df_pbutton_22_23.iloc[10, 3] = df_pbutton_22_23.iloc[[11, 12], 3].sum()

#Somerset
df_pbutton_22_23.iloc[22, 1] = df_pbutton_22_23.iloc[[23, 24], 1].sum()
df_pbutton_22_23.iloc[22, 2] = df_pbutton_22_23.iloc[[23,24], 2].sum()
df_pbutton_22_23.iloc[22, 3] = df_pbutton_22_23.iloc[[23, 24], 3].sum()

#Wicomico
df_pbutton_22_23.iloc[27, 1] = df_pbutton_22_23.iloc[[28, 29], 1].sum()
df_pbutton_22_23.iloc[27, 2] = df_pbutton_22_23.iloc[[28,29], 2].sum()
df_pbutton_22_23.iloc[27, 3] = df_pbutton_22_23.iloc[[28, 29], 3].sum()

# #Worcester
df_pbutton_22_23.iloc[30, 1] = df_pbutton_22_23.iloc[[31, 32], 1].sum()
df_pbutton_22_23.iloc[30, 2] = df_pbutton_22_23.iloc[[31,32], 2].sum()
df_pbutton_22_23.iloc[30, 3] = df_pbutton_22_23.iloc[[31, 32], 3].sum()

In [139]:
#---------2022-2023 Button Buck and Doe Harvest FINAL CLEANING: DROPPING ROWS---------#
#11. Filter county values for "whitetail" and "sika" and drop those rows
df_pbutton_22_23=df_pbutton_22_23[ df_pbutton_22_23[ "County" ].str.contains( "Whitetail|Sika" )== False]
df_pbutton_22_23= df_pbutton_22_23.reset_index(drop=True)
df_pbutton_22_23.to_csv("df_pbutton_22_23.csv")

In [140]:
df_pbutton_22_23

,County,Buttonbuck_2022-23,Female_or_Antlerless_2022-23,Total_2022-23
0,Allegany,138.0,1336.0,1474.0
1,Anne Arundel,149.0,934.0,1083.0
2,Baltimore,353.0,2637.0,2990.0
3,Calvert,126.0,753.0,879.0
4,Caroline,315.0,1653.0,1968.0
5,,315.0,1652.0,1967.0
6,,0.0,1.0,1.0
7,Carroll,433.0,3001.0,3434.0
8,Cecil,283.0,1839.0,2122.0
9,Charles,189.0,1250.0,1439.0


In [141]:
#---------2022-2023 Button Buck and Doe Harvest FINAL CLEANING: DROPPING ROWS---------#
#12. One additional step bc Caroline county doesn't have labeled whitetail and sika rows for this table 

#drop rows 5 and 6

df_pbutton_22_23 = df_pbutton_22_23.drop(df_pbutton_22_23.index[[5,6]])
df_pbutton_22_23= df_pbutton_22_23.reset_index(drop=True)

In [142]:
df_pbutton_22_23.to_csv("df_pbutton_22_23.csv")

In [143]:
#2023-2024 Button buck and doe harvest 

# Open a PDF
pdf = PDF('maryland-Big-Game-Report_2023-24.pdf')
page = pdf.pages[5]
page.show(width=700)

table = page.extract_table(method="pdfplumber")
list(table)

rows = list(table)

df_pbutton_23_24 = pd.DataFrame(rows)

df_pbutton_23_24.to_csv("df_pbutton_23_24.csv", index=False)

In [144]:
#--------2023-2024 Button Buck and Doe Harvest Cleaning Pt. 1--------#

#1. RENAME COLUMNS BY INDEX
#https://www.geeksforgeeks.org/python/rename-column-by-index-in-pandas/

df_pbutton_23_24.rename(columns={
    df_pbutton_23_24.columns[0]: "County",
    df_pbutton_23_24.columns[1]: "Buttonbuck_2023-24",
    df_pbutton_23_24.columns[2]: "Female_or_Antlerless_2023-24",
    df_pbutton_23_24.columns[3]: "Total_2023-24",
    df_pbutton_23_24.columns[4]: "Percent_Buttonbuck"
    
}, inplace=True)
 
#2. FORMAT OF COUNTY NAMES:
    #strip \n
    #Anne Arundel
    #Prince George's 
    #Queen Anne's
df_pbutton_23_24['County']=df_pbutton_23_24['County'].str.replace("\n", " ")
#https://stackoverflow.com/questions/65666852/why-cant-i-replace-a-newline-in-my-pandas-dataframe
#https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

#3. Drop percent in buttonbuck
df_pbutton_23_24 = df_pbutton_23_24.drop(df_pbutton_23_24.columns[4], axis=1)

In [145]:
#--------2023-2024 Button Buck and Doe Harvest Cleaning Pt. 2--------#

#4. Replace asterisks and spaces with NaN
df_pbutton_23_24=df_pbutton_23_24.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

#5. Drop last row, which just has "small sample size" text
df_pbutton_23_24 = df_pbutton_23_24.iloc[:-1]

#6. Drop first row
df_pbutton_23_24.drop(labels=0, inplace=True)

#7. Strip commas
df_pbutton_23_24.replace(',','', regex=True, inplace=True)

,County,Buttonbuck_2023-24,Female_or_Antlerless_2023-24,Total_2023-24
1,Allegany,99,1029,1128
2,Anne Arundel,131,843,974
3,Baltimore,286,2365,2651
4,Calvert,114,703,817
5,Caroline,,,
6,Whitetail,246,1420,1666
7,Sika,,1,1
8,Carroll,439,2820,3259
9,Cecil,203,1641,1844
10,Charles,157,1226,1383


In [146]:
#--------2023-2024 Button Buck and Doe Harvest Cleaning Pt. 3--------#

#8. Change datatype from string to float for numeric avlues 

df_pbutton_23_24['Buttonbuck_2023-24'] = pd.to_numeric( df_pbutton_23_24['Buttonbuck_2023-24'], errors="coerce")
df_pbutton_23_24['Female_or_Antlerless_2023-24'] = pd.to_numeric( df_pbutton_23_24['Female_or_Antlerless_2023-24'], errors="coerce")
df_pbutton_23_24['Total_2023-24'] = pd.to_numeric( df_pbutton_23_24['Total_2023-24'], errors="coerce")

#9. Reset index after dropping rows. 
df_pbutton_23_24 = df_pbutton_23_24.reset_index(drop=True) 

In [147]:
df_pbutton_23_24

,County,Buttonbuck_2023-24,Female_or_Antlerless_2023-24,Total_2023-24
0,Allegany,99.0,1029.0,1128.0
1,Anne Arundel,131.0,843.0,974.0
2,Baltimore,286.0,2365.0,2651.0
3,Calvert,114.0,703.0,817.0
4,Caroline,NaN,NaN,NaN
5,Whitetail,246.0,1420.0,1666.0
6,Sika,NaN,1.0,1.0
7,Carroll,439.0,2820.0,3259.0
8,Cecil,203.0,1641.0,1844.0
9,Charles,157.0,1226.0,1383.0


In [148]:
#--------2023-2024 Button Buck and Doe Harvest Cleaning Pt. 4--------#

#10. Sum whitetail and sika, then move those sums to county row 

#CAROLINE COUNTY
df_pbutton_23_24.iloc[4, 1] = df_pbutton_23_24.iloc[[5, 6], 1].sum()
df_pbutton_23_24.iloc[4, 2] = df_pbutton_23_24.iloc[[5,6], 2].sum()
df_pbutton_23_24.iloc[4, 3] = df_pbutton_23_24.iloc[[5, 6], 3].sum()

# #Dorchester
df_pbutton_23_24.iloc[10, 1] = df_pbutton_23_24.iloc[[11, 12], 1].sum()
df_pbutton_23_24.iloc[10, 2] = df_pbutton_23_24.iloc[[11,12], 2].sum()
df_pbutton_23_24.iloc[10, 3] = df_pbutton_23_24.iloc[[11, 12], 3].sum()

#Somerset
df_pbutton_23_24.iloc[21, 1] = df_pbutton_23_24.iloc[[22, 23], 1].sum()
df_pbutton_23_24.iloc[21, 2] = df_pbutton_23_24.iloc[[22,23], 2].sum()
df_pbutton_23_24.iloc[21, 3] = df_pbutton_23_24.iloc[[22, 23], 3].sum()

#Wicomico
df_pbutton_23_24.iloc[27, 1] = df_pbutton_23_24.iloc[[28, 29], 1].sum()
df_pbutton_23_24.iloc[27, 2] = df_pbutton_23_24.iloc[[28,29], 2].sum()
df_pbutton_23_24.iloc[27, 3] = df_pbutton_23_24.iloc[[28, 29], 3].sum()

# #Worcester
df_pbutton_23_24.iloc[30, 1] = df_pbutton_23_24.iloc[[31, 32], 1].sum()
df_pbutton_23_24.iloc[30, 2] = df_pbutton_23_24.iloc[[31,32], 2].sum()
df_pbutton_23_24.iloc[30, 3] = df_pbutton_23_24.iloc[[31, 32], 3].sum()

In [149]:
#---------2023-2024 Button Buck and Doe Harvest FINAL CLEANING: DROPPING ROWS---------#
#11. Filter county values for "whitetail" and "sika" and drop those rows
df_pbutton_23_24=df_pbutton_23_24[ df_pbutton_23_24[ "County" ].str.contains( "Whitetail|Sika" )== False]
df_pbutton_23_24= df_pbutton_23_24.reset_index(drop=True)
df_pbutton_23_24.to_csv("df_pbutton_23_24.csv")

In [150]:
#2024-2025 Button buck and doe harvest 

# Open a PDF
pdf = PDF('maryland-Big-Game-Report_2024-25.pdf')
page = pdf.pages[7]
page.show(width=700)


table = page.extract_table(method="pdfplumber")
list(table)

rows = list(table)

df_pbutton_24_25 = pd.DataFrame(rows)

df_pbutton_24_25.to_csv("df_pbutton_24_25.csv", index=False)

In [151]:
df_pbutton_24_25

,0,1,2,3,4
0,County,Buttonbuck,Female or\nantlerless,Total,% Buttonbuck
1,Allegany,144,"1,400","1,544",9.3
2,Anne Arundel,189,"1,182","1,371",13.8
3,Baltimore,307,"2,823","3,130",9.8
4,Calvert,122,940,"1,062",11.5
5,Caroline,,,,
6,White-Tailed,314,"1,948","2,262",13.9
7,Sika,0,2,2,0
8,Carroll,437,"3,183","3,620",12.1
9,Cecil,283,"1,977","2,260",12.5


In [152]:
#--------2024-2025 Button Buck and Doe Harvest Cleaning Pt. 1--------#

#1. RENAME COLUMNS BY INDEX
#https://www.geeksforgeeks.org/python/rename-column-by-index-in-pandas/

df_pbutton_24_25.rename(columns={
    df_pbutton_24_25.columns[0]: "County",
    df_pbutton_24_25.columns[1]: "Buttonbuck_2024-25",
    df_pbutton_24_25.columns[2]: "Female_or_Antlerless_2024-25",
    df_pbutton_24_25.columns[3]: "Total_2024-25",
    df_pbutton_24_25.columns[4]: "Percent_Buttonbuck"
    
}, inplace=True)
 
#2. FORMAT OF COUNTY NAMES:
    #strip \n
    #Anne Arundel
    #Prince George's 
    #Queen Anne's
df_pbutton_24_25['County']=df_pbutton_24_25['County'].str.replace("\n", " ")
#https://stackoverflow.com/questions/65666852/why-cant-i-replace-a-newline-in-my-pandas-dataframe
#https://stackoverflow.com/questions/70893764/how-to-change-the-column-type-of-all-columns-except-the-first-in-pandas

#3. Drop percent in buttonbuck
df_pbutton_24_25 = df_pbutton_24_25.drop(df_pbutton_24_25.columns[4], axis=1)

In [153]:
df_pbutton_24_25

,County,Buttonbuck_2024-25,Female_or_Antlerless_2024-25,Total_2024-25
0,County,Buttonbuck,Female or\nantlerless,Total
1,Allegany,144,"1,400","1,544"
2,Anne Arundel,189,"1,182","1,371"
3,Baltimore,307,"2,823","3,130"
4,Calvert,122,940,"1,062"
5,Caroline,,,
6,White-Tailed,314,"1,948","2,262"
7,Sika,0,2,2
8,Carroll,437,"3,183","3,620"
9,Cecil,283,"1,977","2,260"


In [154]:
#--------2024-2025 Button Buck and Doe Harvest Cleaning Pt. 2--------#

#4. Replace asterisks and spaces with NaN
df_pbutton_24_25=df_pbutton_24_25.replace(r'^\*$', np.nan, regex=True)
##https://stackoverflow.com/questions/45311262/replace-a-variable-number-of-asterisks-with-nan-in-a-dataframe 

In [155]:
df_pbutton_24_25

,County,Buttonbuck_2024-25,Female_or_Antlerless_2024-25,Total_2024-25
0,County,Buttonbuck,Female or\nantlerless,Total
1,Allegany,144,"1,400","1,544"
2,Anne Arundel,189,"1,182","1,371"
3,Baltimore,307,"2,823","3,130"
4,Calvert,122,940,"1,062"
5,Caroline,,,
6,White-Tailed,314,"1,948","2,262"
7,Sika,0,2,2
8,Carroll,437,"3,183","3,620"
9,Cecil,283,"1,977","2,260"


In [156]:
#4. LAST ROW ENDS WITH TOTAL FOR 2024-2025 REPORT, NOT NOTE ABOUT SMALL SAMPLE SIZE, SO DON'T HAVE TO DROP.SKIP 

#6. Drop first row
df_pbutton_24_25.drop(labels=0, inplace=True)

#7. Strip commas
df_pbutton_24_25.replace(',','', regex=True, inplace=True)

,County,Buttonbuck_2024-25,Female_or_Antlerless_2024-25,Total_2024-25
1,Allegany,144,1400,1544
2,Anne Arundel,189,1182,1371
3,Baltimore,307,2823,3130
4,Calvert,122,940,1062
5,Caroline,,,
6,White-Tailed,314,1948,2262
7,Sika,0,2,2
8,Carroll,437,3183,3620
9,Cecil,283,1977,2260
10,Charles,208,1804,2012


In [157]:
#--------2016-2017 Button Buck and Doe Harvest Cleaning Pt. 3--------#

#8. Change datatype from string to float for numeric avlues 

df_pbutton_24_25['Buttonbuck_2024-25'] = pd.to_numeric( df_pbutton_24_25['Buttonbuck_2024-25'], errors="coerce")
df_pbutton_24_25['Female_or_Antlerless_2024-25'] = pd.to_numeric( df_pbutton_24_25['Female_or_Antlerless_2024-25'], errors="coerce")
df_pbutton_24_25['Total_2024-25'] = pd.to_numeric( df_pbutton_24_25['Total_2024-25'], errors="coerce")

#9. Reset index after dropping rows. 
df_pbutton_24_25 = df_pbutton_24_25.reset_index(drop=True) 

In [158]:
df_pbutton_24_25

,County,Buttonbuck_2024-25,Female_or_Antlerless_2024-25,Total_2024-25
0,Allegany,144.0,1400.0,1544.0
1,Anne Arundel,189.0,1182.0,1371.0
2,Baltimore,307.0,2823.0,3130.0
3,Calvert,122.0,940.0,1062.0
4,Caroline,NaN,NaN,NaN
5,White-Tailed,314.0,1948.0,2262.0
6,Sika,0.0,2.0,2.0
7,Carroll,437.0,3183.0,3620.0
8,Cecil,283.0,1977.0,2260.0
9,Charles,208.0,1804.0,2012.0


In [159]:
#--------2024-2025 Button Buck and Doe Harvest Cleaning Pt. 4--------#

#10. Sum whitetail and sika, then move those sums to county row 

#CAROLINE COUNTY
df_pbutton_24_25.iloc[4, 1] = df_pbutton_24_25.iloc[[5, 6], 1].sum()
df_pbutton_24_25.iloc[4, 2] = df_pbutton_24_25.iloc[[5,6], 2].sum()
df_pbutton_24_25.iloc[4, 3] = df_pbutton_24_25.iloc[[5, 6], 3].sum()

# #Dorchester
df_pbutton_24_25.iloc[10, 1] = df_pbutton_24_25.iloc[[11, 12], 1].sum()
df_pbutton_24_25.iloc[10, 2] = df_pbutton_24_25.iloc[[11,12], 2].sum()
df_pbutton_24_25.iloc[10, 3] = df_pbutton_24_25.iloc[[11, 12], 3].sum()

#Somerset
df_pbutton_24_25.iloc[21, 1] = df_pbutton_24_25.iloc[[22, 23], 1].sum()
df_pbutton_24_25.iloc[21, 2] = df_pbutton_24_25.iloc[[22,23], 2].sum()
df_pbutton_24_25.iloc[21, 3] = df_pbutton_24_25.iloc[[22, 23], 3].sum()

#Wicomico
df_pbutton_24_25.iloc[27, 1] = df_pbutton_24_25.iloc[[28, 29], 1].sum()
df_pbutton_24_25.iloc[27, 2] = df_pbutton_24_25.iloc[[28,29], 2].sum()
df_pbutton_24_25.iloc[27, 3] = df_pbutton_24_25.iloc[[28, 29], 3].sum()

# #Worcester
df_pbutton_24_25.iloc[30, 1] = df_pbutton_24_25.iloc[[31, 32], 1].sum()
df_pbutton_24_25.iloc[30, 2] = df_pbutton_24_25.iloc[[31,32], 2].sum()
df_pbutton_24_25.iloc[30, 3] = df_pbutton_24_25.iloc[[31, 32], 3].sum()

In [160]:
#---------2024-2025 Button Buck and Doe Harvest FINAL CLEANING: DROPPING ROWS---------#
#11. Filter county values for "whitetail" and "sika" and drop those rows
df_pbutton_24_25=df_pbutton_24_25[ df_pbutton_24_25[ "County" ].str.contains( "White-Tailed|Sika" )== False]
df_pbutton_24_25= df_pbutton_24_25.reset_index(drop=True)
df_pbutton_24_25.to_csv("df_pbutton_24_25.csv")

In [161]:
df_pbutton_24_25

,County,Buttonbuck_2024-25,Female_or_Antlerless_2024-25,Total_2024-25
0,Allegany,144.0,1400.0,1544.0
1,Anne Arundel,189.0,1182.0,1371.0
2,Baltimore,307.0,2823.0,3130.0
3,Calvert,122.0,940.0,1062.0
4,Caroline,314.0,1950.0,2264.0
5,Carroll,437.0,3183.0,3620.0
6,Cecil,283.0,1977.0,2260.0
7,Charles,208.0,1804.0,2012.0
8,Dorchester,337.0,3260.0,3597.0
9,Frederick,375.0,3482.0,3857.0


In [162]:
#---------COMBINE BUTTON BUCK AND DOE THROUGH CONCAT, STANDARDIZE LABELS, DOUBLE-CHECK FOR CONSISTENT SPELLING OF COUNTY NAMES---------#

df_pbutton_combined_harvest = pd.concat([df_pbutton_15_16, df_pbutton_16_17, df_pbutton_17_18, df_pbutton_18_19, df_pbutton_19_20,
           df_pbutton_20_21, df_pbutton_21_22, df_pbutton_22_23, df_pbutton_23_24, df_pbutton_24_25])

df_pbutton_combined_harvest.to_csv("df_pbutton_combined_harvest.csv")